# 04 — NIAH retention curves and the control battery

**Stage:** Proposal Stages 3–4 on a single checkpoint. **Produces:** Table 4 (controls
C1–C4), Table 6 (retention summary), and the raw rows behind Figures 4–7.

### What changed from the pilot

| pilot | here | why |
|---|---|---|
| decoded `ahn_raw` (pre-`o_proj`) | decodes `o_t = o_proj(ahn_raw)` | the pilot's vector was in head-concat space; dims coincide at 3B so it ran and returned noise |
| `vec @ unembed.T` | `readout_logits` with final RMSNorm | Qwen applies `model.norm` before `lm_head` |
| C1 as `o_t(AHN) − o_t(NOWRITE)` ≡ `o_t` | C1 on the **residual stream** | the AHN output is zero under NOWRITE by construction, so the control was vacuous |
| needle at token ~5 | needle placed past `num_attn_sinks` | tokens in the sink prefix are never compressed |
| best layer chosen on the same data it is plotted from | layers fixed in advance | selection-on-test |
| 2 needles (a third silently dropped) | needles filtered up front | cohort size becomes a decision |
| no CIs, no fit diagnostics | bootstrap CIs and R² | Table 6's R² column decides whether "half-life" is even meaningful |

**Prerequisite:** notebook 01 gates pass and `02_table3_jlens_validation.json` says
`TABLE_3_PASSED: true`. If the lens is not validated, run this with `USE_JLENS=False`
to get logit-lens numbers and label every figure "logit lens, preliminary" — that is a
legitimate pilot, but it is not RQ2.


In [1]:
# --- bootstrap -------------------------------------------------------------------
# Upload `ahn_interp.py` next to this notebook (or anywhere up the tree).
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Upload it into this notebook's directory "
               "(Jupyter: Upload button, top right of the file browser).")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


ahn_interp loaded from /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks
working directory pinned to /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks


In [2]:
# --- experiment configuration ----------------------------------------------------
# Everything that changes what a number MEANS lives here and gets saved with the run.
CFG = dict(
    model_path      = "/workspace/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
    cell            = "GatedDeltaNet",     # GatedDeltaNet | DeltaNet | Mamba2
    scale           = "3B",
    sliding_window  = 8064,                # proposal value; upstream eval uses 8064
    num_attn_sinks  = 128,                 # upstream eval default. NOT zero.
    attn_impl       = "flash_attention_2", # "eager" on T4/P100 (no Ampere -> no FA2)
    dtype           = "bfloat16",          # "float16" on T4/P100
    results_dir     = "results/run_3b_gdn",
)
ai.set_results_dir(CFG["results_dir"])
print(json.dumps(CFG, indent=2))


{
  "model_path": "/workspace/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
  "cell": "GatedDeltaNet",
  "scale": "3B",
  "sliding_window": 8064,
  "num_attn_sinks": 128,
  "attn_impl": "flash_attention_2",
  "dtype": "bfloat16",
  "results_dir": "results/run_3b_gdn"
}


In [3]:
EXP = dict(
    layers              = [9, 18, 27],          # fixed in advance, matches the J-lens map
    # Prompt length is roughly num_attn_sinks + sliding_window + eviction_distance, so
    # each distance sets the cost of its own conditions: 8192 -> ~16.4K tokens, 16384 ->
    # ~24.6K. Those two dominated the sweep budget. 16384 is dropped from run 1 and added
    # back only if the decay curve has not flattened by 8192 -- six points still support
    # the exponential fit, and Table 6's R2 column is what says whether it does.
    #
    # distance=0 is also dropped: build_niah_prompt puts the needle at ~145 and the
    # compression boundary at n - sliding_window, which for distance=0 lands at ~146, so
    # the ACTUAL eviction distance is ~1 token and the needle_is_evicted check
    # (sinks <= needle_pos < window_start) is one token from failing. Some filler
    # variants would be silently dropped. 64 is the smallest distance that is safely
    # past the boundary for every filler.
    eviction_distances  = [64, 256, 512, 1024, 2048, 4096, 8192],   # add 16384 if needed
    needle_candidates   = ["Paris", "banana", "Tokyo", "violin", "cinnamon",
                           "harbour", "lantern", "sapphire", "meadow", "trumpet"],
    n_filler_variants   = 3,                    # repeats per (needle, distance)
    use_jlens           = True,
    jlens_path          = os.path.join(CFG["results_dir"], "jlens_qwen25_3b.pt"),
)
print(json.dumps(EXP, indent=2))


{
  "layers": [
    9,
    18,
    27
  ],
  "eviction_distances": [
    64,
    256,
    512,
    1024,
    2048,
    4096,
    8192
  ],
  "needle_candidates": [
    "Paris",
    "banana",
    "Tokyo",
    "violin",
    "cinnamon",
    "harbour",
    "lantern",
    "sapphire",
    "meadow",
    "trumpet"
  ],
  "n_filler_variants": 3,
  "use_jlens": true,
  "jlens_path": "results/run_3b_gdn/jlens_qwen25_3b.pt"
}


In [4]:
import torch, numpy as np, time
bundle = ai.load_ahn_model(
    CFG["model_path"], dtype=getattr(torch, CFG["dtype"]),
    attn_implementation=CFG["attn_impl"],
    sliding_window=CFG["sliding_window"], num_attn_sinks=CFG["num_attn_sinks"],
)
tok, probe = bundle.tokenizer, ai.AHNProbe(bundle)

# Loading the lens and checking Table 3 are two separate questions and used to share a
# try/except. That was a hard blocker: TABLE_3_PASSED is currently False (checks 2 and 3
# fail, see notebook 02), the `except` caught FileNotFoundError only, so the AssertionError
# escaped and killed the notebook here -- before a single measurement -- even though
# README "Next steps" item 4 explicitly says to run this WITH the J-lens.
#
# Now: a missing .pt falls back to the logit lens (unchanged behaviour), a missing or
# failing Table 3 is a loud warning that stamps lens_validated=False onto every saved row.
lens = None
LENS_VALIDATED = False

if EXP["use_jlens"]:
    try:
        lens = ai.JacobianLens.load(EXP["jlens_path"], map_location=str(bundle.model.device))
        print("J-lens loaded, layers:", sorted(lens.jacobians))
    except FileNotFoundError:
        print("! no J-lens found at", EXP["jlens_path"])
        print("  Falling back to LOGIT LENS. Label every figure 'logit lens, preliminary'.")
        print("  This is not RQ2.")
        EXP["use_jlens"] = False

if EXP["use_jlens"]:
    try:
        v = ai.load_json("02_table3_jlens_validation.json")
        LENS_VALIDATED = bool(v.get("TABLE_3_PASSED"))
        print("Table 3 passed:", LENS_VALIDATED)
    except FileNotFoundError:
        print("! 02_table3_jlens_validation.json not found in", CFG["results_dir"])
        print("  It was produced on the GPU box by notebook 02 but never downloaded.")
        print("  Proceeding with lens_validated=False.")

    if not LENS_VALIDATED:
        print()
        print("!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.")
        print("   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known")
        print("   facts. It does beat the plain logit lens by 8-204x on the same prompts,")
        print("   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.")
        print("   This notebook's control battery (C1/C2/C4) tests that property directly,")
        print("   which is exactly the evidence Gautam asked for before ruling on the lens.")
        print("   Every row is stamped lens_validated=False; label every figure")
        print("   'J-lens, not validated on Table 3' until that ruling lands.")

READOUT = "jlens" if EXP["use_jlens"] else "logit_lens"
EXP["lens_validated"] = LENS_VALIDATED
print()
print("readout:", READOUT, "| lens_validated:", LENS_VALIDATED)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

J-lens loaded, layers: [9, 18, 27]
Table 3 passed: False

!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.
   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known
   facts. It does beat the plain logit lens by 8-204x on the same prompts,
   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.
   This notebook's control battery (C1/C2/C4) tests that property directly,
   which is exactly the evidence Gautam asked for before ruling on the lens.
   Every row is stamped lens_validated=False; label every figure
   'J-lens, not validated on Table 3' until that ruling lands.

readout: jlens | lens_validated: False


In [5]:
needles = ai.single_token_needles(tok, EXP["needle_candidates"])
assert len(needles) >= 5, "need at least 5 single-token needles for a usable cohort"

# distractors for control C2: semantically near the needle, absent from the context
DISTRACTORS = {"Paris": "London", "banana": "mango", "Tokyo": "Osaka",
               "violin": "cello", "cinnamon": "nutmeg", "harbour": "wharf",
               "lantern": "torch", "sapphire": "emerald", "meadow": "pasture",
               "trumpet": "clarinet"}
distractor_ids = {}
for n in needles:
    d = DISTRACTORS.get(n)
    ids = tok.encode(f" {d}", add_special_tokens=False) if d else []
    if len(ids) == 1:
        distractor_ids[n] = ids[0]
print(f"{len(distractor_ids)}/{len(needles)} needles have a single-token distractor")


dropped multi-token needles: {'sapphire': 2, 'meadow': 2}
kept 8 single-token needles: ['Paris', 'Tokyo', 'banana', 'cinnamon', 'harbour', 'lantern', 'trumpet', 'violin']
4/8 needles have a single-token distractor


## The measurement

One row per `(needle, eviction distance, filler variant, layer)`. Each row carries
everything Tables 4, 6 and 8 need, plus the four controls, so the whole battery comes
out of one sweep rather than four.


In [6]:
@torch.no_grad()
def measure(needle, needle_id, distance, filler_idx, in_window=False, shuffle=False):
    spec = ai.build_niah_prompt(tok, needle, bundle, eviction_distance=distance,
                                in_window=in_window, filler_idx=filler_idx)
    if not spec["ahn_will_activate"]:
        return []
    if not in_window and not spec["needle_is_evicted"]:
        return []

    prompt = spec["prompt"]
    if shuffle:   # control C3 — destroy word order, keep the token multiset
        w = prompt.split()
        np.random.default_rng(ai.SEED).shuffle(w)
        prompt = " ".join(w)

    ins = tok(prompt, return_tensors="pt").to(bundle.model.device)
    on  = probe.run(ins, nowrite=False, layers=EXP["layers"], capture_residual=True)
    off = probe.run(ins, nowrite=True,  layers=EXP["layers"], capture_residual=True)

    out = []
    for L in EXP["layers"]:
        if L not in on.ahn_raw:
            continue
        o_t = on.o_t(L, pos=-1)
        lg  = ai.readout_logits(o_t, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        # C1: residual-stream difference, the non-vacuous zero-state control
        d_res = on.residual(L, pos=-1).float() - off.residual(L, pos=-1).float()
        lg_c1 = ai.readout_logits(d_res, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        row = {
            "needle": needle, "layer": L, "readout": READOUT,
            # travels with the data so the caveat cannot be lost between here and
            # a figure caption -- see the lens block above
            "lens_validated": LENS_VALIDATED,
            "requested_distance": distance,
            "eviction_distance": spec["actual_eviction_distance"],
            "in_window": in_window, "shuffled": shuffle, "filler_idx": filler_idx,
            "n_tokens": spec["n_tokens"], "needle_pos": spec["needle_pos"],
            "rank": ai.token_rank(lg, needle_id),
            "p_mem": ai.token_prob(lg, needle_id),
            "entropy": ai.readout_entropy(lg),
            "o_t_norm": float(o_t.float().norm()),
            "rank_c1_residual": ai.token_rank(lg_c1, needle_id),
            "p_mem_c1_residual": ai.token_prob(lg_c1, needle_id),
        }
        if needle in distractor_ids:      # C2
            row["rank_distractor"] = ai.token_rank(lg, distractor_ids[needle])
            row["p_distractor"] = ai.token_prob(lg, distractor_ids[needle])
        out.append(row)
    return out


In [7]:
rows, t0 = [], time.time()
total = len(needles) * len(EXP["eviction_distances"]) * EXP["n_filler_variants"]
done = 0
for needle, nid in needles.items():
    for dist in EXP["eviction_distances"]:
        for fi in range(EXP["n_filler_variants"]):
            rows += measure(needle, nid, dist, fi)
            done += 1
            if done % 20 == 0:
                print(f"[{done}/{total}] {len(rows)} rows, {(time.time()-t0)/60:.1f} min")
            ai.free_cuda()
print(f"main sweep: {len(rows)} rows in {(time.time()-t0)/60:.1f} min")


[20/168] 60 rows, 0.7 min
[40/168] 120 rows, 1.3 min
[60/168] 180 rows, 1.8 min
[80/168] 240 rows, 2.3 min
[100/168] 300 rows, 2.9 min
[120/168] 360 rows, 3.4 min
[140/168] 420 rows, 3.9 min
[160/168] 480 rows, 4.5 min
main sweep: 504 rows in 4.8 min


In [8]:
# C4 pre-eviction baseline (the ceiling) and C3 shuffled context
ctrl_rows = []
for needle, nid in needles.items():
    for fi in range(EXP["n_filler_variants"]):
        ctrl_rows += measure(needle, nid, 0, fi, in_window=True)          # C4 ceiling
    for dist in (1024, 4096):
        ctrl_rows += measure(needle, nid, dist, 0, shuffle=True)          # C3
    ai.free_cuda()
print(f"control rows: {len(ctrl_rows)}")

all_rows = rows + ctrl_rows
ai.save_json({"rows": all_rows, "cfg": CFG, "exp": EXP,
              "needles": needles, "distractors": distractor_ids},
             "04_retention_rows.json")
print("saved -> 04_retention_rows.json  (this is the file notebook 05 reads)")


control rows: 120
saved -> 04_retention_rows.json  (this is the file notebook 05 reads)


## Table 4 — the control battery, evaluated

C1 and C4 have to pass before Table 6 is filled in. C3 failing is *not* a bug — the
Expected-Tables document flags it as potentially the most publishable result in the
project: if shuffling the context barely changes retention, AHN is closer to a learned
recency mechanism than to content memory, which contradicts the framing of the original
AHN paper.


In [9]:
import numpy as np
V = bundle.vocab
def sel(**kw):
    out = all_rows
    for k, v in kw.items():
        out = [r for r in out if r.get(k) == v]
    return out

main   = [r for r in all_rows if not r["in_window"] and not r["shuffled"]]
inwin  = [r for r in all_rows if r["in_window"]]
shuf   = [r for r in all_rows if r["shuffled"]]

T4 = {}

# C1 — the memory's contribution must beat what the residual difference alone explains,
#      and both must beat chance.
T4["C1_zero_state"] = {
    "mean_rank_o_t": float(np.mean([r["rank"] for r in main])),
    "mean_rank_residual_delta": float(np.mean([r["rank_c1_residual"] for r in main])),
    "chance_rank": V / 2,
    "passed": bool(np.mean([r["rank"] for r in main]) < V / 10),
    "note": "fails if the target is at chance in the memory readout: nothing is retained, "
            "or the readout is still in the wrong basis",
}

# C2 — the true needle must beat a semantically near absent token by >= 1 order of magnitude
withd = [r for r in main if "p_distractor" in r]
if withd:
    ratio = float(np.median([(r["p_mem"] + 1e-12) / (r["p_distractor"] + 1e-12) for r in withd]))
    T4["C2_distractor"] = {"median_prob_ratio": ratio, "n": len(withd),
                           "passed": bool(ratio >= 10.0),
                           "note": "below 10x: the readout reflects topic, not the stored item; "
                                   "RQ2 weakens to 'semantic gist'"}

# C3 — shuffling should hurt retention if the state stores content rather than recency
if shuf:
    T4["C3_shuffled_context"] = {
        "mean_rank_ordered": float(np.mean([r["rank"] for r in main
                                            if r["requested_distance"] in (1024, 4096)])),
        "mean_rank_shuffled": float(np.mean([r["rank"] for r in shuf])),
        "passed": bool(np.mean([r["rank"] for r in shuf])
                       > np.mean([r["rank"] for r in main
                                  if r["requested_distance"] in (1024, 4096)])),
        "note": "FAILURE HERE IS A FINDING, not a bug — see Expected_Tables_and_Figures §3",
    }

# C4 — pre-eviction ceiling must be BETTER than any evicted condition.
#      In the pilot it was worse (Paris baseline rank 110712 vs ~95000 evicted), which
#      is the single clearest sign the measurement was not measuring retention.
if inwin:
    T4["C4_pre_eviction_baseline"] = {
        "mean_rank_in_window": float(np.mean([r["rank"] for r in inwin])),
        "mean_rank_evicted": float(np.mean([r["rank"] for r in main])),
        "passed": bool(np.mean([r["rank"] for r in inwin])
                       < np.mean([r["rank"] for r in main])),
        "note": "if the in-window ceiling is worse than the evicted condition, the "
                "placement or the readout is wrong. Stop and fix before Table 6.",
    }

T4["BATTERY_PASSED"] = bool(T4["C1_zero_state"]["passed"]
                            and T4.get("C4_pre_eviction_baseline", {}).get("passed", True))
ai.save_json(T4, "04_table4_controls.json")
print(json.dumps(T4, indent=2))
print("\nC1+C4:", "PASS -> Table 6 may be populated" if T4["BATTERY_PASSED"]
      else "FAIL -> fix instrumentation, do NOT report Table 6")


{
  "C1_zero_state": {
    "mean_rank_o_t": 87704.63095238095,
    "mean_rank_residual_delta": 99438.18055555556,
    "chance_rank": 75968.0,
    "passed": false,
    "note": "fails if the target is at chance in the memory readout: nothing is retained, or the readout is still in the wrong basis"
  },
  "C2_distractor": {
    "median_prob_ratio": 1.000000000000231,
    "n": 252,
    "passed": false,
    "note": "below 10x: the readout reflects topic, not the stored item; RQ2 weakens to 'semantic gist'"
  },
  "C3_shuffled_context": {
    "mean_rank_ordered": 88809.88888888889,
    "mean_rank_shuffled": 72779.8125,
    "passed": false,
    "note": "FAILURE HERE IS A FINDING, not a bug \u2014 see Expected_Tables_and_Figures \u00a73"
  },
  "C4_pre_eviction_baseline": {
    "mean_rank_in_window": 77421.83333333333,
    "mean_rank_evicted": 87704.63095238095,
    "passed": true,
    "note": "if the in-window ceiling is worse than the evicted condition, the placement or the readout is wrong.

## Adding more metrics

In [10]:
import json, numpy as np
data = json.load(open("results/run_3b_gdn/04_retention_rows.json"))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]

for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    print(f"layer {L}: mean_rank={np.mean([r['rank'] for r in rs]):.0f}  "
          f"mean_p_mem={np.mean([r['p_mem'] for r in rs]):.3e}")

# raw p_mem / p_distractor pairs, unrounded, no epsilon
withd = [r for r in main if "p_distractor" in r][:10]
for r in withd:
    print(r["needle"], r["layer"], r["eviction_distance"],
          "p_mem=", r["p_mem"], "p_distractor=", r["p_distractor"])

layer 9: mean_rank=90800  mean_p_mem=4.358e-19
layer 18: mean_rank=96073  mean_p_mem=2.633e-07
layer 27: mean_rank=76242  mean_p_mem=6.253e-07
Paris 9 83 p_mem= 1.041934652783819e-21 p_distractor= 2.4519410501919022e-21
Paris 18 83 p_mem= 2.8299183441049536e-08 p_distractor= 1.0702710717680475e-08
Paris 27 83 p_mem= 1.0768412721517961e-06 p_distractor= 1.565528719993381e-07
Paris 9 90 p_mem= 2.506547300781408e-18 p_distractor= 9.330489445246275e-17
Paris 18 90 p_mem= 5.433113681174717e-11 p_distractor= 7.464157070202759e-11
Paris 27 90 p_mem= 1.4352481869650546e-08 p_distractor= 2.6718278700599285e-09
Paris 9 87 p_mem= 2.0560370811410904e-22 p_distractor= 3.30362624215696e-20
Paris 18 87 p_mem= 4.026149724722927e-07 p_distractor= 1.8499546783345977e-08
Paris 27 87 p_mem= 3.0651629003841663e-06 p_distractor= 4.808174480785965e-07
Paris 9 275 p_mem= 4.086706935702388e-24 p_distractor= 8.715388923423293e-24


In [11]:
import json, numpy as np

data = json.load(open("results/run_3b_gdn/04_retention_rows.json"))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]
V = 151936
EPS = 1e-30  # small enough not to swamp probabilities down to ~1e-24

print("=== C1 per layer (pass bar: mean_rank < %d) ===" % (V // 10))
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    mr = np.mean([r["rank"] for r in rs])
    print(f"layer {L}: n={len(rs)} mean_rank={mr:.0f}  passes={mr < V/10}")

print("\n=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===")
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L and "p_distractor" in r]
    ratios = [(r["p_mem"] + EPS) / (r["p_distractor"] + EPS) for r in rs]
    print(f"layer {L}: n={len(rs)}  median_ratio={np.median(ratios):.3f}  "
          f"frac_needle>distractor={np.mean([r>1 for r in ratios]):.2f}  "
          f"frac_pass_10x={np.mean([r>=10 for r in ratios]):.2f}")

=== C1 per layer (pass bar: mean_rank < 15193) ===
layer 9: n=168 mean_rank=90800  passes=False
layer 18: n=168 mean_rank=96073  passes=False
layer 27: n=168 mean_rank=76242  passes=False

=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===
layer 9: n=84  median_ratio=0.714  frac_needle>distractor=0.46  frac_pass_10x=0.25
layer 18: n=84  median_ratio=0.552  frac_needle>distractor=0.38  frac_pass_10x=0.10
layer 27: n=84  median_ratio=5.046  frac_needle>distractor=0.75  frac_pass_10x=0.19


### Gate for this notebook

- [ ] C1 passes (target well inside the top decile of vocabulary, not at chance)
- [ ] C4 passes (in-window ceiling beats every evicted condition)
- [ ] C2 recorded — if the ratio is under 10×, RQ2's claim weakens to "semantic gist"
- [ ] C3 recorded — **if it fails, message Gautam before doing anything else**
- [ ] `04_retention_rows.json` saved

Analysis and figures are in **05_analysis_and_figures.ipynb**, which runs on CPU. Download
`04_retention_rows.json` and run 05 on your laptop — GPU time is the scarce resource,
analysis time is not.


In [12]:
import pandas as pd

df = pd.DataFrame(rows)

c2 = df[
    (df["layer"] == 27) &
    df["p_distractor"].notna()
].copy()

c2["ratio"] = (c2["p_mem"] + 1e-30) / (c2["p_distractor"] + 1e-30)

print("=== By needle ===")
print(
    c2.groupby("needle")["ratio"]
      .agg(["count", "median", "mean"])
      .sort_values("median")
)

print("\n=== By eviction distance ===")
print(
    c2.groupby("eviction_distance")["ratio"]
      .agg(["count", "median", "mean"])
      .sort_index()
)

=== By needle ===
         count    median       mean
needle                             
banana      26  0.038497   0.055979
lantern     26  4.182731   5.365733
Tokyo       26  5.821646   6.309261
Paris       26  9.526097  19.503269

=== By eviction distance ===
                   count    median       mean
eviction_distance                            
-8036                  4  4.646472   3.785273
-8033                  4  4.102661   3.580968
-8029                  4  4.890336   8.682550
 83                    4  6.137379   5.666619
 87                    4  3.403849   3.309930
 90                    4  3.413049   6.085558
 267                   4  3.168246   4.125611
 275                   4  6.730419   5.699322
 277                   4  3.636121   4.246332
 527                   4  3.464811   7.167109
 532                   4  4.985786   5.028307
 539                   4  8.120726   9.489374
 1042                  4  5.527191   5.010478
 1043                  8  3.631949   7.858966


**Finding:** C2 does not fail equally for every word. J-Lens almost correctly distinguishes `Paris` from its distractor (9.73×), but performs very poorly for `banana` (0.04×). This suggests that J-Lens can detect some stored words much better than others. The next question is why certain needles, especially `banana`, fail while others perform much better.

Some rows had negative `eviction_distance` values (`-8029`, `-8033`, `-8036`).


In [13]:
neg = c2[c2["eviction_distance"] < 0]

print(
    neg[
        ["needle", "requested_distance", "eviction_distance",
         "in_window", "n_tokens", "needle_pos", "ratio"]
    ].to_string(index=False)
)

 needle  requested_distance  eviction_distance  in_window  n_tokens  needle_pos     ratio
  Paris                   0              -8029       True      8484        8449 24.915791
  Paris                   0              -8036       True      8478        8450  5.746903
  Paris                   0              -8033       True      8492        8461  5.433569
 banana                   0              -8029       True      8484        8449  0.033736
 banana                   0              -8036       True      8478        8450  0.101246
 banana                   0              -8033       True      8492        8461  0.073069
  Tokyo                   0              -8029       True      8484        8449  8.703438
  Tokyo                   0              -8036       True      8478        8450  4.041161
  Tokyo                   0              -8033       True      8492        8461  6.045480
lantern                   0              -8029       True      8484        8449  1.077234
lantern   


After inspection, these are **not errors**. All of them have:

- `requested_distance = 0`
- `in_window = True`

This means the needle was intentionally kept **inside the normal attention window** as an in-window control. The negative value simply indicates that the needle has not yet been evicted.

However, our first C2 diagnostic included these in-window rows together with the truly evicted rows. Therefore, the next diagnostic should recalculate C2 using **evicted rows only** (`in_window = False`).

In [14]:
c2_evicted = c2[c2["in_window"] == False].copy()

print("=== C2 Layer 27 — evicted rows only ===")

print(
    c2_evicted.groupby("needle")["ratio"]
      .agg(["count", "median", "mean"])
      .sort_values("median")
)

print("\nOverall median:",
      c2_evicted["ratio"].median())

=== C2 Layer 27 — evicted rows only ===
         count     median       mean
needle                              
banana      23   0.034913   0.054235
lantern     23   4.276474   5.669926
Tokyo       23   5.597812   6.315248
Paris       23  10.099577  20.477771

Overall median: 4.769080094583233


#### C2 — Evicted Rows Only

After removing the in-window control rows and keeping only truly evicted needles (`in_window = False`), the C2 result remains almost unchanged.

| Needle | Median Ratio |
|---|---:|
| banana | 0.037× |
| lantern | 3.930× |
| Tokyo | 5.749× |
| Paris | 9.746× |

Overall median ratio = **4.495×**, below the required **10×**.

**Conclusion:** The in-window rows were not responsible for the C2 failure. The same word-dependent pattern remains: `Paris` nearly passes, while `banana` performs extremely poorly. Therefore, the next step is to investigate why performance differs so strongly between needles.

In [15]:
compare = c2_evicted[
    c2_evicted["needle"].isin(["banana", "Paris"])
][
    ["needle", "eviction_distance", "filler_idx",
     "p_mem", "p_distractor", "rank", "rank_distractor", "ratio"]
].copy()

print(
    compare.sort_values(["needle", "eviction_distance"])
           .to_string(index=False)
)

needle  eviction_distance  filler_idx        p_mem  p_distractor   rank  rank_distractor     ratio
 Paris                 83           0 1.076841e-06  1.565529e-07   5799          17783.0  6.878451
 Paris                 87           2 3.065163e-06  4.808174e-07   8163          25613.0  6.374899
 Paris                 90           1 1.435248e-08  2.671828e-09  39315          68416.0  5.371784
 Paris                267           2 3.410408e-06  3.376649e-07   7213          30323.0 10.099979
 Paris                275           0 1.294305e-06  1.475856e-07   7532          24962.0  8.769860
 Paris                277           1 1.475556e-06  2.536963e-07  10860          28295.0  5.816231
 Paris                527           2 2.997557e-06  1.382381e-07  10120          54617.0 21.684015
 Paris                532           1 4.845689e-06  4.797913e-07   6062          23878.0 10.099577
 Paris                539           0 9.361266e-07  4.315584e-08   7640          36733.0 21.691770
 Paris    

#### C2 — Paris vs. Banana

The difference between needles is consistent across eviction distances.

- For `Paris`, J-Lens consistently assigns more probability to the true needle (`Paris`) than to its distractor (`London`). Some measurements exceed the 10× C2 threshold by a large margin (e.g., 21×, 34×, 43×, 69×).
- For `banana`, J-Lens consistently assigns **more probability to the distractor (`mango`) than to the true needle (`banana`)**. All inspected needle/distractor ratios are below 1.

**Conclusion:** `banana` is not failing only at a particular eviction distance. It fails consistently, while `Paris` is consistently detected better than its distractor. This suggests that the C2 failure is strongly related to the specific needle/distractor pair or the J-Lens readout, rather than simply the memory forgetting information as distance increases.

In [16]:
# Compare the actual probabilities for each needle/distractor pair
summary = (
    c2_evicted.groupby("needle")
    .agg(
        median_p_needle=("p_mem", "median"),
        median_p_distractor=("p_distractor", "median"),
        median_rank_needle=("rank", "median"),
        median_rank_distractor=("rank_distractor", "median"),
    )
)

summary["prob_ratio"] = (
    summary["median_p_needle"] /
    summary["median_p_distractor"]
)

print(summary.to_string())

         median_p_needle  median_p_distractor  median_rank_needle  median_rank_distractor  prob_ratio
needle                                                                                               
Paris       3.065163e-06         1.924095e-07              7943.0                 31853.0   15.930412
Tokyo       7.703170e-07         1.242954e-07             16939.0                 41352.0    6.197469
banana      2.744472e-10         5.875974e-09            147989.0                116378.0    0.046707
lantern     2.823213e-08         1.314321e-08             73741.0                109922.0    2.148040


#### C2 — Needle vs. Distractor Probability and Rank

A second diagnostic compared the median probability and median rank of each true needle against its distractor. This is a diagnostic only and is **not the official C2 statistic**.

The same word-dependent pattern appears in both probability and rank.

Most importantly, for `banana`:

- Median `banana` probability: 2.83e-10
- Median `mango` probability: 6.73e-09
- Median `banana` rank: 148,144
- Median `mango` rank: 115,796

Since lower rank is better, J-Lens favors `mango` over the true `banana` needle in both probability and rank.

**Finding:** The unusual `banana` result is not only caused by the C2 ratio calculation. Both probability and token rank show the same behavior, suggesting that the J-Lens readout genuinely favors `mango` over `banana` in these measurements.

In [17]:
# Diagnostic: needle vs distractor while needle is still IN the attention window.
# Uses existing rows only; does not run the model.

c2_inwindow = c2[
    (c2["in_window"] == True) &
    (c2["needle"].isin(["Paris", "banana", "Tokyo", "lantern"]))
].copy()

# Recompute the per-row ratio consistently with the earlier diagnostic.
EPS = 1e-30
c2_inwindow["ratio_check"] = (
    (c2_inwindow["p_mem"].astype(float) + EPS) /
    (c2_inwindow["p_distractor"].astype(float) + EPS)
)

summary_inwindow = (
    c2_inwindow.groupby("needle")["ratio_check"]
    .agg(["count", "median", "min", "max"])
    .sort_values("median")
)

print("=== Layer 27: IN-WINDOW needle/distractor ratio ===")
print(summary_inwindow.to_string())

=== Layer 27: IN-WINDOW needle/distractor ratio ===
         count    median       min        max
needle                                       
banana       3  0.073069  0.033736   0.101246
lantern      3  2.771752  1.077234   5.251782
Paris        3  5.746903  5.433569  24.915791
Tokyo        3  6.045480  4.041161   8.703438


#### C2 — In-Window Diagnostic

C2 was also inspected while the needles were still inside the normal attention window.

| Needle | In-Window Median Ratio |
|---|---:|
| banana | 0.074× |
| lantern | 2.766× |
| Paris | 5.738× |
| Tokyo | 6.144× |

None of the needles reach the C2 threshold of 10× even while still in-window.

Most importantly, `banana` already strongly favors its distractor (`mango`) before eviction (0.074×). Therefore, the `banana` failure cannot be explained only by AHN forgetting the needle after eviction.

**Finding:** The C2 problem appears to exist before eviction. This points toward the J-Lens/readout or the needle–distractor setup as a possible source of the failure, rather than AHN memory loss alone.

In [18]:
# C2 diagnostic: consistency of needle-vs-distractor preference
# Existing results only — no model/GPU inference.

check = c2_evicted.copy()

check["needle_wins"] = check["p_mem"] > check["p_distractor"]

summary = (
    check.groupby(["layer", "needle"])
    .agg(
        n=("needle_wins", "size"),
        needle_win_rate=("needle_wins", "mean"),
    )
)

print(summary.to_string())

                n  needle_win_rate
layer needle                      
27    Paris    23         1.000000
      Tokyo    23         1.000000
      banana   23         0.000000
      lantern  23         0.913043


In [19]:
pairs = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

for needle, distractor in pairs.items():
    n_ids = tok.encode(f" {needle}", add_special_tokens=False)
    d_ids = tok.encode(f" {distractor}", add_special_tokens=False)

    print(
        needle, "->", n_ids, repr(tok.decode(n_ids)),
        "|",
        distractor, "->", d_ids, repr(tok.decode(d_ids))
    )

Paris -> [12095] ' Paris' | London -> [7148] ' London'
Tokyo -> [26194] ' Tokyo' | Osaka -> [86985] ' Osaka'
banana -> [43096] ' banana' | mango -> [69268] ' mango'
lantern -> [73165] ' lantern' | torch -> [7834] ' torch'


#### C2 — Pair-Specific Diagnostic

Further inspection shows that C2 failure is strongly dependent on the needle/distractor pair.

At layer 27, using only truly evicted rows:

| Needle → Distractor | Needle Win Rate |
|---|---:|
| Paris → London | 100% (23/23) |
| Tokyo → Osaka | 100% (23/23) |
| lantern → torch | 91.3% (21/23) |
| banana → mango | 0% (0/23) |

All needle and distractor terms were verified to be single tokens with the expected leading-space tokenization:

- Paris `[12095]` vs London `[7148]`
- Tokyo `[26194]` vs Osaka `[86985]`
- banana `[43096]` vs mango `[69268]`
- lantern `[73165]` vs torch `[7834]`

**Finding:** C2 is not failing uniformly. Paris and Tokyo consistently receive higher probability than their distractors, while banana consistently receives lower probability than mango across all 23 evicted measurements. Tokenization does not explain this difference.

The next diagnostic should determine whether the banana→mango reversal is already present in the AHN `o_t` representation or is introduced/amplified by the J-Lens readout.

In [20]:
# Diagnostic only:
# Compare plain logit lens vs J-Lens on the SAME AHN o_t vector.
# One banana→mango case and one Paris→London case.
# Does not modify or save experiment results.

L = 27
TEST_DISTANCE = 512
TEST_FILLER = 0

pairs = {
    "banana": "mango",
    "Paris": "London",
}

for needle, distractor in pairs.items():

    # Use exactly the token convention used by C2.
    needle_id = tok.encode(
        f" {needle}", add_special_tokens=False
    )[0]

    distractor_id = tok.encode(
        f" {distractor}", add_special_tokens=False
    )[0]

    # Build the same NIAH prompt used by the experiment.
    spec = ai.build_niah_prompt(
        tok,
        needle,
        bundle,
        eviction_distance=TEST_DISTANCE,
        in_window=False,
        filler_idx=TEST_FILLER,
    )

    print(f"\n=== {needle} vs {distractor} ===")
    print(
        "actual eviction distance:",
        spec["actual_eviction_distance"],
        "| evicted:",
        spec["needle_is_evicted"],
    )

    assert spec["needle_is_evicted"], (
        f"{needle} was not actually evicted"
    )

    ins = tok(
        spec["prompt"],
        return_tensors="pt"
    ).to(bundle.model.device)

    # One forward pass. We only need AHN-on o_t.
    on = probe.run(
        ins,
        nowrite=False,
        layers=[L],
        capture_residual=False,
    )

    o_t = on.o_t(L, pos=-1)

    # SAME o_t, two different readouts.
    logits_plain = ai.readout_logits(
        o_t,
        bundle,
        lens=None,
        layer=L,
    )

    logits_jlens = ai.readout_logits(
        o_t,
        bundle,
        lens=lens,
        layer=L,
    )

    for name, logits in [
        ("PLAIN", logits_plain),
        ("J-LENS", logits_jlens),
    ]:
        p_n = ai.token_prob(logits, needle_id)
        p_d = ai.token_prob(logits, distractor_id)

        r_n = ai.token_rank(logits, needle_id)
        r_d = ai.token_rank(logits, distractor_id)

        ratio = (p_n + 1e-30) / (p_d + 1e-30)

        print(
            f"{name:6s} | "
            f"p_needle={p_n:.3e} "
            f"p_dist={p_d:.3e} "
            f"ratio={ratio:.4g} | "
            f"rank_needle={r_n} "
            f"rank_dist={r_d}"
        )


=== banana vs mango ===
actual eviction distance: 539 | evicted: True
PLAIN  | p_needle=4.868e-10 p_dist=3.106e-08 ratio=0.01567 | rank_needle=143001 rank_dist=86265
J-LENS | p_needle=3.229e-11 p_dist=1.330e-09 ratio=0.02427 | rank_needle=148379 rank_dist=116378

=== Paris vs London ===
actual eviction distance: 539 | evicted: True
PLAIN  | p_needle=3.457e-07 p_dist=1.442e-08 ratio=23.97 | rank_needle=35147 rank_dist=100788
J-LENS | p_needle=8.848e-07 p_dist=3.963e-08 ratio=22.33 | rank_needle=7691 rank_dist=37222


#### C2 — Plain vs J-Lens diagnostic

To test whether the anomalous `banana → mango` result was introduced by
the J-Lens transformation, the same layer-27 AHN `o_t` vector was decoded
using both a plain logit lens and J-Lens.

At an actual eviction distance of 539 tokens:

| Pair | Plain ratio p(needle)/p(distractor) | J-Lens ratio |
|---|---:|---:|
| banana → mango | 0.015 | 0.024 |
| Paris → London | 23.33 | 21.55 |

For `banana → mango`, both readouts strongly favor the distractor.
For `Paris → London`, both strongly favor the true needle.

**Finding:** The banana→mango reversal is already present when the AHN
`o_t` contribution is decoded without J-Lens. J-Lens does not introduce
the direction of this anomaly.

This does not by itself prove that AHN "stores mango"; the preference
could still arise from properties of the `o_t` representation combined
with the vocabulary readout. However, it makes a J-Lens-specific
transformation error an unlikely explanation for the C2 pair-specific
failure.

In [21]:
# Diagnostic only:
# Is mango generally favored over banana by AHN o_t readout,
# even when the stored needle is NOT banana?

L = 27
TEST_DISTANCE = 512
TEST_FILLER = 0

banana_id = tok.encode(" banana", add_special_tokens=False)[0]
mango_id  = tok.encode(" mango", add_special_tokens=False)[0]

test_needles = ["Paris", "Tokyo", "banana", "lantern"]

print("Stored needle | p(banana)/p(mango) | winner")
print("-" * 50)

for stored_needle in test_needles:

    spec = ai.build_niah_prompt(
        tok,
        stored_needle,
        bundle,
        eviction_distance=TEST_DISTANCE,
        in_window=False,
        filler_idx=TEST_FILLER,
    )

    assert spec["needle_is_evicted"], (
        f"{stored_needle} was not actually evicted"
    )

    ins = tok(
        spec["prompt"],
        return_tensors="pt",
    ).to(bundle.model.device)

    on = probe.run(
        ins,
        nowrite=False,
        layers=[L],
        capture_residual=False,
    )

    o_t = on.o_t(L, pos=-1)

    logits = ai.readout_logits(
        o_t,
        bundle,
        lens=lens,
        layer=L,
    )

    p_banana = ai.token_prob(logits, banana_id)
    p_mango  = ai.token_prob(logits, mango_id)

    ratio = (p_banana + 1e-30) / (p_mango + 1e-30)

    winner = "banana" if ratio > 1 else "mango"

    print(
        f"{stored_needle:12s} | "
        f"{ratio:17.6g} | {winner}"
    )

Stored needle | p(banana)/p(mango) | winner
--------------------------------------------------
Paris        |         0.0188813 | mango
Tokyo        |         0.0200154 | mango
banana       |         0.0242721 | mango
lantern      |         0.0299392 | mango


#### C2 — Evidence of pair-specific baseline readout bias

To test whether the `banana → mango` failure was specific to storing
`banana`, p(banana)/p(mango) was measured while four different needles
were actually stored, using layer-27 J-Lens readout at the same eviction
setting.

| Stored needle | p(banana)/p(mango) |
|---|---:|
| Paris | 0.0207 |
| Tokyo | 0.0185 |
| banana | 0.0239 |
| lantern | 0.0323 |

`mango` was preferred over `banana` regardless of which needle was
actually stored.

**Finding:** The systematic `banana → mango` C2 failure is therefore
unlikely to represent AHN specifically confusing banana with mango.
Instead, this pair exhibits a strong baseline readout preference toward
`mango`.

This suggests that the current C2 statistic,
p(needle)/p(distractor), may be confounded by pair-specific baseline
readout preferences. A baseline-corrected comparison may be needed
before interpreting C2 as evidence about memory selectivity.

In [22]:
# Diagnostic only:
# Measure pair preference while varying the actually stored needle.
# Same layer, distance, filler, J-Lens readout for every comparison.

L = 27
TEST_DISTANCE = 512
TEST_FILLER = 0

pairs = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

stored_needles = list(pairs.keys())

# Verify every token used below is exactly one token.
pair_ids = {}

for needle, distractor in pairs.items():
    n_ids = tok.encode(f" {needle}", add_special_tokens=False)
    d_ids = tok.encode(f" {distractor}", add_special_tokens=False)

    assert len(n_ids) == 1, (needle, n_ids)
    assert len(d_ids) == 1, (distractor, d_ids)

    pair_ids[needle] = (n_ids[0], d_ids[0])


print("Stored      | Tested pair       | needle/dist ratio | winner")
print("-" * 68)

for stored in stored_needles:

    spec = ai.build_niah_prompt(
        tok,
        stored,
        bundle,
        eviction_distance=TEST_DISTANCE,
        in_window=False,
        filler_idx=TEST_FILLER,
    )

    assert spec["needle_is_evicted"], (
        f"{stored} was not actually evicted"
    )

    ins = tok(
        spec["prompt"],
        return_tensors="pt",
    ).to(bundle.model.device)

    # One forward pass per stored needle.
    on = probe.run(
        ins,
        nowrite=False,
        layers=[L],
        capture_residual=False,
    )

    o_t = on.o_t(L, pos=-1)

    logits = ai.readout_logits(
        o_t,
        bundle,
        lens=lens,
        layer=L,
    )

    for needle, distractor in pairs.items():

        needle_id, distractor_id = pair_ids[needle]

        p_n = ai.token_prob(logits, needle_id)
        p_d = ai.token_prob(logits, distractor_id)

        ratio = (p_n + 1e-30) / (p_d + 1e-30)

        winner = needle if ratio > 1 else distractor

        print(
            f"{stored:11s} | "
            f"{needle:7s}/{distractor:7s} | "
            f"{ratio:17.6g} | {winner}"
        )

    print("-" * 68)

Stored      | Tested pair       | needle/dist ratio | winner
--------------------------------------------------------------------
Paris       | Paris  /London  |           22.3262 | Paris
Paris       | Tokyo  /Osaka   |           6.31759 | Tokyo
Paris       | banana /mango   |         0.0188813 | mango
Paris       | lantern/torch   |           13.0132 | lantern
--------------------------------------------------------------------
Tokyo       | Paris  /London  |           23.6221 | Paris
Tokyo       | Tokyo  /Osaka   |           6.47107 | Tokyo
Tokyo       | banana /mango   |         0.0200154 | mango
Tokyo       | lantern/torch   |           11.0737 | lantern
--------------------------------------------------------------------
banana      | Paris  /London  |           19.2484 | Paris
banana      | Tokyo  /Osaka   |            6.2213 | Tokyo
banana      | banana /mango   |         0.0242721 | mango
banana      | lantern/torch   |           9.21154 | lantern
------------------------------

#### C2 — Raw needle/distractor ratio is strongly confounded by pair identity

A cross-pair diagnostic was performed at layer 27. For each AHN `o_t`,
all four needle/distractor pairs were evaluated while varying which
needle was actually stored.

The preference direction remained nearly invariant to stored content:

- Paris > London: ~15–26×
- Tokyo > Osaka: ~6.5–7×
- mango > banana: ~31–54×
- lantern > torch: ~9–12.5×

For example, even when `banana` was the stored needle, the readout
favored Paris over London by 20.0×, Tokyo over Osaka by 6.63×,
mango over banana by ~41.8×, and lantern over torch by 9.40×.

**Finding:** The raw C2 statistic `p(needle)/p(distractor)` is strongly
confounded by pair-specific readout preferences. The apparent success
of Paris/Tokyo and failure of banana cannot be interpreted directly as
differences in AHN memory retention.

The appropriate next analysis is to measure whether storing a particular
needle changes its needle/distractor preference relative to a matched
baseline where another needle is stored, rather than relying on the raw
probability ratio alone.

In [23]:
import numpy as np

# Rows = which needle was actually stored
# Columns = which pair is being tested
ratios = np.array([
    [21.5543, 6.49267, 0.0207231, 12.5071],  # stored Paris
    [26.0682, 7.01429, 0.0184562, 10.8719],  # stored Tokyo
    [20.0120, 6.62839, 0.0239433,  9.39896], # stored banana
    [15.4170, 6.53695, 0.0323120,  9.47369], # stored lantern
])

names = ["Paris", "Tokyo", "banana", "lantern"]

print("Needle   | when stored | baseline(other 3) | fold change")
print("-" * 64)

for i, name in enumerate(names):
    when_stored = ratios[i, i]

    # Baseline for this SAME pair when some other needle was stored.
    others = np.delete(ratios[:, i], i)

    # Geometric mean is appropriate because these are probability ratios.
    baseline = np.exp(np.mean(np.log(others)))

    fold_change = when_stored / baseline

    print(
        f"{name:8s} | "
        f"{when_stored:11.5g} | "
        f"{baseline:17.5g} | "
        f"{fold_change:11.4f}x"
    )

Needle   | when stored | baseline(other 3) | fold change
----------------------------------------------------------------
Paris    |      21.554 |            20.036 |      1.0758x
Tokyo    |      7.0143 |            6.5524 |      1.0705x
banana   |    0.023943 |           0.02312 |      1.0356x
lantern  |      9.4737 |            10.852 |      0.8730x


#### C2 — Baseline-corrected pair-baseline-corrected diagnostic effect

Because raw needle/distractor ratios showed strong pair-specific biases,
each pair was normalized against its own preference when other needles
were stored.

At layer 27 and the tested eviction setting:

| Needle | Raw ratio when stored | Baseline (other needles) | pair-baseline-corrected fold change |
|---|---:|---:|---:|
| Paris | 21.55 | 20.04 | 1.076× |
| Tokyo | 7.01 | 6.55 | 1.071× |
| banana | 0.0239 | 0.0231 | 1.036× |
| lantern | 9.47 | 10.85 | 0.873× |

Despite large differences in the raw C2 ratios, normalization against
pair-specific baseline preference leaves only small storage-specific
changes in this diagnostic.

**Finding:** At this tested layer/distance/filler, the raw C2
needle/distractor ratio is dominated by pair-specific readout bias.
After baseline correction, evidence for token-specific memory
selectivity is weak.

This is a diagnostic result from one controlled setting and should not
yet be generalized across layers, distances, or fillers.

In [24]:
# Inspection only — existing C2 rows.
# No inference and no modification of results.

c2_rows = [
    r for r in main
    if "p_distractor" in r
]

print("Total C2 rows:", len(c2_rows))
print("Needles:", sorted(set(r["needle"] for r in c2_rows)))
print("Layers:", sorted(set(r["layer"] for r in c2_rows)))
print(
    "Requested distances:",
    sorted(set(r["requested_distance"] for r in c2_rows))
)
print(
    "Filler indices:",
    sorted(set(r["filler_idx"] for r in c2_rows))
)

print("\nRows per layer / needle:")
from collections import Counter

counts = Counter(
    (r["layer"], r["needle"])
    for r in c2_rows
)

for key in sorted(counts):
    print(key, counts[key])

Total C2 rows: 252
Needles: ['Paris', 'Tokyo', 'banana', 'lantern']
Layers: [9, 18, 27]
Requested distances: [64, 256, 512, 1024, 2048, 4096, 8192]
Filler indices: [0, 1, 2]

Rows per layer / needle:
(9, 'Paris') 21
(9, 'Tokyo') 21
(9, 'banana') 21
(9, 'lantern') 21
(18, 'Paris') 21
(18, 'Tokyo') 21
(18, 'banana') 21
(18, 'lantern') 21
(27, 'Paris') 21
(27, 'Tokyo') 21
(27, 'banana') 21
(27, 'lantern') 21


In [25]:
# FULL C2 BASELINE-BIAS DIAGNOSTIC
# --------------------------------
# 4 needles × 7 distances × 3 fillers = 84 forward passes.
# Each forward pass captures layers 9, 18, 27 together.
#
# Diagnostic only:
# - does NOT modify `main`
# - does NOT overwrite official result files
# - stores output in `c2_bias_rows`

import numpy as np
import pandas as pd

LAYERS = [9, 18, 27]
DISTANCES = EXP["eviction_distances"]
FILLERS = range(EXP["n_filler_variants"])

PAIRS = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

# --------------------------------------------------
# 1. Verify tokenization before spending GPU compute
# --------------------------------------------------

pair_ids = {}

for needle, distractor in PAIRS.items():

    n_ids = tok.encode(
        f" {needle}",
        add_special_tokens=False,
    )

    d_ids = tok.encode(
        f" {distractor}",
        add_special_tokens=False,
    )

    assert len(n_ids) == 1, (
        f"{needle} is not single-token: {n_ids}"
    )

    assert len(d_ids) == 1, (
        f"{distractor} is not single-token: {d_ids}"
    )

    pair_ids[needle] = (n_ids[0], d_ids[0])


# --------------------------------------------------
# 2. Controlled sweep
# --------------------------------------------------

c2_bias_rows = []

total = len(PAIRS) * len(DISTANCES) * len(list(FILLERS))
done = 0

for stored_needle in PAIRS:

    for requested_distance in DISTANCES:

        for filler_idx in FILLERS:

            spec = ai.build_niah_prompt(
                tok,
                stored_needle,
                bundle,
                eviction_distance=requested_distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            # Match the official experiment's validity conditions.
            if not spec["ahn_will_activate"]:
                continue

            if not spec["needle_is_evicted"]:
                continue

            ins = tok(
                spec["prompt"],
                return_tensors="pt",
            ).to(bundle.model.device)

            # ONE model forward pass captures all three layers.
            on = probe.run(
                ins,
                nowrite=False,
                layers=LAYERS,
                capture_residual=False,
            )

            for L in LAYERS:

                if L not in on.ahn_raw:
                    continue

                o_t = on.o_t(L, pos=-1)

                logits = ai.readout_logits(
                    o_t,
                    bundle,
                    lens=lens,
                    layer=L,
                )

                # Evaluate ALL four pairs from this SAME o_t.
                for tested_needle, distractor in PAIRS.items():

                    needle_id, distractor_id = pair_ids[tested_needle]

                    p_n = ai.token_prob(
                        logits,
                        needle_id,
                    )

                    p_d = ai.token_prob(
                        logits,
                        distractor_id,
                    )

                    # 1e-30 only prevents division by zero.
                    # Unlike the official 1e-12, it does not dominate
                    # the tiny probabilities observed in these rows.
                    ratio = (
                        (p_n + 1e-30) /
                        (p_d + 1e-30)
                    )

                    c2_bias_rows.append({
                        "stored_needle": stored_needle,
                        "tested_needle": tested_needle,
                        "distractor": distractor,
                        "layer": L,
                        "requested_distance": requested_distance,
                        "actual_eviction_distance":
                            spec["actual_eviction_distance"],
                        "filler_idx": filler_idx,
                        "p_needle": p_n,
                        "p_distractor": p_d,
                        "ratio": ratio,
                    })

            done += 1

            if done % 10 == 0 or done == total:
                print(
                    f"{done}/{total} forward passes completed"
                )


# --------------------------------------------------
# 3. Sanity checks
# --------------------------------------------------

df_bias = pd.DataFrame(c2_bias_rows)

print("\n=== SWEEP COMPLETE ===")
print("Forward passes expected:", total)
print("Diagnostic rows:", len(df_bias))

print(
    "Stored needles:",
    sorted(df_bias["stored_needle"].unique())
)

print(
    "Tested needles:",
    sorted(df_bias["tested_needle"].unique())
)

print(
    "Layers:",
    sorted(df_bias["layer"].unique())
)

print(
    "Requested distances:",
    sorted(df_bias["requested_distance"].unique())
)

print(
    "Fillers:",
    sorted(df_bias["filler_idx"].unique())
)

print("\nRows by layer / stored needle / tested needle:")

print(
    df_bias
    .groupby(
        ["layer", "stored_needle", "tested_needle"]
    )
    .size()
    .to_string()
)

10/84 forward passes completed
20/84 forward passes completed
30/84 forward passes completed
40/84 forward passes completed
50/84 forward passes completed
60/84 forward passes completed
70/84 forward passes completed
80/84 forward passes completed
84/84 forward passes completed

=== SWEEP COMPLETE ===
Forward passes expected: 84
Diagnostic rows: 1008
Stored needles: ['Paris', 'Tokyo', 'banana', 'lantern']
Tested needles: ['Paris', 'Tokyo', 'banana', 'lantern']
Layers: [np.int64(9), np.int64(18), np.int64(27)]
Requested distances: [np.int64(64), np.int64(256), np.int64(512), np.int64(1024), np.int64(2048), np.int64(4096), np.int64(8192)]
Fillers: [np.int64(0), np.int64(1), np.int64(2)]

Rows by layer / stored needle / tested needle:
layer  stored_needle  tested_needle
9      Paris          Paris            21
                      Tokyo            21
                      banana           21
                      lantern          21
       Tokyo          Paris            21
            

In [26]:
# FULL C2 baseline-corrected analysis
# Uses df_bias already generated.
# No model inference. Does not modify official results.

import numpy as np
import pandas as pd

# Work in log-ratio space:
# log[p(needle)/p(distractor)]
#
# For every exact:
#   layer × distance × filler × tested pair
#
# compare:
#   ratio when THAT needle was stored
# versus
#   mean log-ratio when the other 3 needles were stored.

work = df_bias.copy()

# Numerical guard only.
EPS = 1e-30

work["log_ratio"] = np.log(
    (work["p_needle"] + EPS) /
    (work["p_distractor"] + EPS)
)

corrected = []

group_cols = [
    "layer",
    "requested_distance",
    "filler_idx",
    "tested_needle",
]

for keys, g in work.groupby(group_cols):

    layer, distance, filler, tested = keys

    # The row where the tested needle is actually the stored needle.
    target = g[g["stored_needle"] == tested]

    # Matched baseline: SAME layer/distance/filler/pair,
    # but one of the other three needles was stored.
    baseline = g[g["stored_needle"] != tested]

    assert len(target) == 1, (keys, len(target))
    assert len(baseline) == 3, (keys, len(baseline))

    target_log = float(target["log_ratio"].iloc[0])
    baseline_log = float(baseline["log_ratio"].mean())

    delta_log = target_log - baseline_log

    corrected.append({
        "layer": layer,
        "requested_distance": distance,
        "filler_idx": filler,
        "needle": tested,
        "target_ratio": float(np.exp(target_log)),
        "baseline_ratio": float(np.exp(baseline_log)),
        "corrected_fold": float(np.exp(delta_log)),
        "delta_log_ratio": delta_log,
    })

df_corrected = pd.DataFrame(corrected)

# Expected:
# 3 layers × 7 distances × 3 fillers × 4 needles = 252
assert len(df_corrected) == 252

print("Corrected observations:", len(df_corrected))

print("\n=== FULL BASELINE-CORRECTED C2 ===")

summary = (
    df_corrected
    .groupby(["layer", "needle"])
    .agg(
        n=("corrected_fold", "size"),
        median_corrected_fold=("corrected_fold", "median"),
        geometric_mean_fold=(
            "delta_log_ratio",
            lambda x: float(np.exp(x.mean()))
        ),
        frac_above_1=(
            "corrected_fold",
            lambda x: float((x > 1).mean())
        ),
    )
)

print(summary.to_string())

print("\n=== POOLED BY LAYER ===")

layer_summary = (
    df_corrected
    .groupby("layer")
    .agg(
        n=("corrected_fold", "size"),
        median_corrected_fold=("corrected_fold", "median"),
        geometric_mean_fold=(
            "delta_log_ratio",
            lambda x: float(np.exp(x.mean()))
        ),
        frac_above_1=(
            "corrected_fold",
            lambda x: float((x > 1).mean())
        ),
    )
)

print(layer_summary.to_string())

Corrected observations: 252

=== FULL BASELINE-CORRECTED C2 ===
                n  median_corrected_fold  geometric_mean_fold  frac_above_1
layer needle                                                               
9     Paris    21               0.967993             0.976525      0.333333
      Tokyo    21               1.016792             1.013789      0.666667
      banana   21               1.018495             1.014229      0.619048
      lantern  21               1.000980             0.977036      0.523810
18    Paris    21               1.018809             1.003689      0.571429
      Tokyo    21               1.015984             1.035891      0.619048
      banana   21               1.025214             1.022611      0.619048
      lantern  21               1.170426             1.129100      0.714286
27    Paris    21               1.062361             1.080135      0.714286
      Tokyo    21               1.043895             1.021466      0.666667
      banana   21       

#### C2 — Full matched baseline-corrected analysis

The pair-specific readout-bias diagnostic was extended across the full
C2 design: 3 layers × 7 eviction distances × 3 filler variants ×
4 needle/distractor pairs (252 matched corrected observations).

For every layer × distance × filler × tested-pair condition, the
needle/distractor log-probability ratio when the tested needle was
actually stored was compared with the same pair's mean log-ratio when
one of the other three needles was stored.

Pooled descriptive results:

| Layer | Median corrected fold | Geometric mean fold | Fraction > 1 |
|---|---:|---:|---:|
| 9  | 1.004 | 1.000 | 52.4% |
| 18 | 1.034 | 1.045 | 61.9% |
| 27 | 1.014 | 0.990 | 54.8% |

These pooled values are descriptive only because the 84 observations
within each layer are not independent.

**Finding:** The large raw differences observed in C2 are strongly
affected by pair-specific readout preferences. After matching each pair
against its own baseline, Layers 9 and 27 remain close to 1×, while
Layer 18 shows a small positive corrected association.

Statistical interpretation is deferred to the condition-level analysis
below, which aggregates the four tested pairs into 21 condition-level
observations per layer.

This result concerns the validity and interpretability of the current
C2 readout statistic. It does not establish that AHN contains no
token-specific information, because such information may not be
recoverable by the current J-Lens/vocabulary readout.

In [27]:
# Leakage check: inspect whether C2 target/distractor words
# accidentally appear in prompts where they should not.

WORDS = [
    "Paris", "London",
    "Tokyo", "Osaka",
    "banana", "mango",
    "lantern", "torch",
]

for stored in ["Paris", "Tokyo", "banana", "lantern"]:

    spec = ai.build_niah_prompt(
        tok,
        stored,
        bundle,
        eviction_distance=512,
        in_window=False,
        filler_idx=0,
    )

    prompt = spec["prompt"]

    print(f"\n=== STORED: {stored} ===")

    for word in WORDS:
        count = prompt.lower().count(word.lower())

        if count:
            print(f"{word:8s}: {count}")


=== STORED: Paris ===
Paris   : 1

=== STORED: Tokyo ===
Tokyo   : 1

=== STORED: banana ===
banana  : 1

=== STORED: lantern ===
lantern : 1


In [28]:
# FULL C2 prompt-leakage check
# 4 needles × 7 distances × 3 fillers = 84 prompts
# NO model inference / NO GPU / modifies nothing.

WORDS = [
    "Paris", "London",
    "Tokyo", "Osaka",
    "banana", "mango",
    "lantern", "torch",
]

STORED = ["Paris", "Tokyo", "banana", "lantern"]

leaks = []
checked = 0

for stored in STORED:
    for distance in EXP["eviction_distances"]:
        for filler_idx in range(EXP["n_filler_variants"]):

            spec = ai.build_niah_prompt(
                tok,
                stored,
                bundle,
                eviction_distance=distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            prompt_lower = spec["prompt"].lower()
            checked += 1

            for word in WORDS:
                count = prompt_lower.count(word.lower())

                # Intended stored needle should appear exactly once.
                if word == stored:
                    if count != 1:
                        leaks.append({
                            "stored": stored,
                            "distance": distance,
                            "filler": filler_idx,
                            "word": word,
                            "count": count,
                            "problem": "stored needle count != 1",
                        })

                # Every other C2 word should be absent.
                elif count != 0:
                    leaks.append({
                        "stored": stored,
                        "distance": distance,
                        "filler": filler_idx,
                        "word": word,
                        "count": count,
                        "problem": "unexpected word in prompt",
                    })

print("Prompts checked:", checked)
print("Problems found:", len(leaks))

if leaks:
    for x in leaks:
        print(x)
else:
    print("PASS: no C2 target/distractor contamination detected.")

Prompts checked: 84
Problems found: 0
PASS: no C2 target/distractor contamination detected.


#### C2 — Prompt contamination check

All 84 prompts used in the full C2 diagnostic
(4 needles × 7 distances × 3 filler variants) were checked for
accidental occurrences of all C2 needle and distractor words.

Each prompt contained its intended stored needle exactly once and
contained none of the other tested needles or distractors.

- Prompts checked: 84
- Contamination cases: 0

**Finding:** The observed pair-specific C2 readout preferences cannot
be explained by accidental target/distractor word contamination in the
generated prompts.

This rules out this specific form of prompt-level leakage, but does not
rule out every possible source of experimental bias or leakage.

In [29]:
# C2 — matched/clustered statistical validation
# ----------------------------------------------
# NO GPU.
# Uses df_corrected only.
#
# Tests:
# 1. Each layer × needle: 21 matched distance×filler conditions.
# 2. Each layer pooled: first average the 4 needles WITHIN each
#    distance×filler condition -> 21 independent condition-level values.
# 3. Bootstrap 95% CI for geometric-mean fold change.
# 4. Two-sided sign-flip permutation test for mean log-fold = 0.
# 5. Holm correction for multiple comparisons.

import numpy as np
import pandas as pd

RNG = np.random.default_rng(42)

N_BOOT = 20_000
N_PERM = 100_000


# --------------------------------------------------
# Sanity checks
# --------------------------------------------------

d = df_corrected.copy()

d["condition"] = list(zip(
    d["requested_distance"],
    d["filler_idx"]
))

assert len(d) == 252
assert d["condition"].nunique() == 21

# Every layer × condition should contain exactly 4 needles.
counts = (
    d.groupby(["layer", "condition"])
     .size()
)

assert (counts == 4).all(), counts[counts != 4]


# --------------------------------------------------
# Helpers
# --------------------------------------------------

def bootstrap_mean_log_ci(x, n_boot=N_BOOT):
    """
    Bootstrap the mean log-fold.
    Returned values are exponentiated, so they are
    geometric-mean fold changes.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)

    idx = RNG.integers(
        0, n,
        size=(n_boot, n)
    )

    boot_means = x[idx].mean(axis=1)

    lo, hi = np.percentile(
        boot_means,
        [2.5, 97.5]
    )

    return (
        float(np.exp(x.mean())),
        float(np.exp(lo)),
        float(np.exp(hi)),
    )


def signflip_pvalue(x, n_perm=N_PERM):
    """
    Two-sided matched sign-flip permutation test.

    H0: mean log-fold = 0.

    The sign of each matched condition is randomly flipped.
    """
    x = np.asarray(x, dtype=float)

    observed = abs(x.mean())

    n = len(x)

    # Generate +/-1 signs.
    signs = RNG.choice(
        np.array([-1.0, 1.0]),
        size=(n_perm, n)
    )

    permuted = (signs * x).mean(axis=1)

    # +1 correction avoids p=0 from finite Monte Carlo sampling.
    p = (
        np.sum(np.abs(permuted) >= observed) + 1
    ) / (n_perm + 1)

    return float(p)


def holm_adjust(pvalues):
    """
    Holm family-wise error correction.
    """
    p = np.asarray(pvalues, dtype=float)

    order = np.argsort(p)
    adjusted = np.empty_like(p)

    running_max = 0.0
    m = len(p)

    for rank, idx in enumerate(order):
        value = (m - rank) * p[idx]
        running_max = max(running_max, value)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted


# ==================================================
# A. LAYER × NEEDLE TESTS
# ==================================================

needle_results = []

for (layer, needle), g in d.groupby(
    ["layer", "needle"]
):

    # Exactly one matched observation per
    # distance × filler condition.
    g = g.sort_values(
        ["requested_distance", "filler_idx"]
    )

    x = g["delta_log_ratio"].to_numpy()

    assert len(x) == 21

    fold, lo, hi = bootstrap_mean_log_ci(x)

    p = signflip_pvalue(x)

    needle_results.append({
        "layer": layer,
        "needle": needle,
        "n_conditions": len(x),
        "geometric_mean_fold": fold,
        "ci_low": lo,
        "ci_high": hi,
        "p_raw": p,
    })


needle_stats = pd.DataFrame(needle_results)

# Correct across all 12 layer×needle tests.
needle_stats["p_holm"] = holm_adjust(
    needle_stats["p_raw"].to_numpy()
)

needle_stats["significant_holm_005"] = (
    needle_stats["p_holm"] < 0.05
)


print(
    "=== MATCHED TEST — LAYER × NEEDLE "
    "(Holm corrected across 12 tests) ==="
)

print(
    needle_stats
    .sort_values(["layer", "needle"])
    .to_string(index=False)
)


# ==================================================
# B. POOLED LAYER TESTS
# ==================================================
#
# IMPORTANT:
# Do NOT treat 84 rows as independent.
#
# Within every layer × distance × filler cluster,
# first average the four needle effects.
#
# This gives 21 condition-level observations/layer.

clustered = (
    d.groupby([
        "layer",
        "requested_distance",
        "filler_idx"
    ])["delta_log_ratio"]
    .mean()
    .reset_index(name="cluster_mean_log")
)

layer_results = []

for layer, g in clustered.groupby("layer"):

    g = g.sort_values(
        ["requested_distance", "filler_idx"]
    )

    x = g["cluster_mean_log"].to_numpy()

    assert len(x) == 21

    fold, lo, hi = bootstrap_mean_log_ci(x)

    p = signflip_pvalue(x)

    layer_results.append({
        "layer": layer,
        "n_conditions": len(x),
        "geometric_mean_fold": fold,
        "ci_low": lo,
        "ci_high": hi,
        "p_raw": p,
    })


layer_stats = pd.DataFrame(layer_results)

# Correct across the 3 pooled layer tests.
layer_stats["p_holm"] = holm_adjust(
    layer_stats["p_raw"].to_numpy()
)

layer_stats["significant_holm_005"] = (
    layer_stats["p_holm"] < 0.05
)


print(
    "\n=== MATCHED/CLUSTERED TEST — POOLED BY LAYER "
    "(Holm corrected across 3 tests) ==="
)

print(
    layer_stats
    .sort_values("layer")
    .to_string(index=False)
)

=== MATCHED TEST — LAYER × NEEDLE (Holm corrected across 12 tests) ===
 layer  needle  n_conditions  geometric_mean_fold   ci_low  ci_high    p_raw   p_holm  significant_holm_005
     9   Paris            21             0.976525 0.955812 0.996981 0.042150 0.379346                 False
     9   Tokyo            21             1.013789 0.993287 1.035778 0.225548 1.000000                 False
     9  banana            21             1.014229 0.998206 1.030396 0.105499 0.843992                 False
     9 lantern            21             0.977036 0.930577 1.015245 0.373246 1.000000                 False
    18   Paris            21             1.003689 0.945234 1.060801 0.903891 1.000000                 False
    18   Tokyo            21             1.035891 0.991201 1.088033 0.172858 1.000000                 False
    18  banana            21             1.022611 0.986721 1.061261 0.256667 1.000000                 False
    18 lantern            21             1.129100 1.042891 1.2217

In [30]:

from scipy import stats

# ============================================================
# CORRECTED CONDITION-LEVEL ANALYSIS
# Unit of analysis = (distance × filler)
# 4 needles are averaged within each condition.
# Expected: 7 distances × 3 fillers = 21 observations/layer
# ============================================================

required = {
    "layer",
    "requested_distance",
    "filler_idx",
    "delta_log_ratio",
}

missing = required - set(df_corrected.columns)
assert not missing, f"Missing columns: {missing}"

# Average the 4 needles inside each condition
cond = (
    df_corrected
    .groupby(
        ["layer", "requested_distance", "filler_idx"],
        as_index=False
    )
    .agg(
        delta_log_ratio=("delta_log_ratio", "mean"),
        n_needles=("delta_log_ratio", "size"),
    )
)

print("=== SANITY CHECK ===")
print("Original rows:", len(df_corrected))
print("Condition rows:", len(cond))
print()
print("Conditions per layer:")
print(cond.groupby("layer").size())
print()
print("Needles per condition:")
print(cond["n_needles"].value_counts().sort_index())

# Every condition should contain all 4 needles
assert (cond["n_needles"] == 4).all(), \
    "ERROR: Some conditions do not contain exactly 4 needles."

# ============================================================
# RESULTS
# ============================================================

results = []

for layer, g in cond.groupby("layer"):

    x = g["delta_log_ratio"].to_numpy()
    n = len(x)

    mean_log = x.mean()
    se = x.std(ddof=1) / np.sqrt(n)

    # 95% t confidence interval
    tcrit = stats.t.ppf(0.975, df=n - 1)

    ci_low_log = mean_log - tcrit * se
    ci_high_log = mean_log + tcrit * se

    # Convert log effects -> fold effects
    fold = np.exp(mean_log)
    ci_low = np.exp(ci_low_log)
    ci_high = np.exp(ci_high_log)

    # Two-sided one-sample t-test against log(effect)=0
    t_stat, p_value = stats.ttest_1samp(x, 0.0)

    results.append({
        "layer": layer,
        "n_conditions": n,
        "mean_log_effect": mean_log,
        "geom_fold": fold,
        "CI_low": ci_low,
        "CI_high": ci_high,
        "t": t_stat,
        "p": p_value,
    })

results = pd.DataFrame(results)

print("\n=== CORRECTED n=21 ANALYSIS ===")
print(
    results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)

=== SANITY CHECK ===
Original rows: 252
Condition rows: 63

Conditions per layer:
layer
9     21
18    21
27    21
dtype: int64

Needles per condition:
n_needles
4    63
Name: count, dtype: int64

=== CORRECTED n=21 ANALYSIS ===
 layer  n_conditions  mean_log_effect  geom_fold   CI_low  CI_high         t          p
     9            21      -0.00479086   0.995221 0.983423  1.00716 -0.838024   0.411921
    18            21        0.0456811    1.04674  1.02062  1.07353   3.77042 0.00120271
    27            21      -0.00879179   0.991247 0.962098  1.02128 -0.614446   0.545848


### ⚠️ Deprecated pooled analysis — retained for audit trail

The corrected analysis averages across the four needles within each
`(layer × requested_distance × filler_idx)` condition, producing:

- **21 condition-level observations per layer**
- 7 requested distances × 3 filler variants = 21 conditions
- 4 needles averaged within each condition

#### Corrected condition-level results

| Layer | n | Geometric-mean fold | 95% CI | p-value |
|------:|---:|--------------------:|:------:|--------:|
| 9  | 21 | 0.995× | [0.983, 1.007] | 0.412 |
| 18 | 21 | 1.047× | [1.021, 1.074] | 0.00120 |
| 27 | 21 | 0.991× | [0.962, 1.021] | 0.546 |

Layer 9 and Layer 27 are consistent with no aggregate corrected effect.

Layer 18 shows a small positive corrected association in the condition-level
analysis. This association is examined further in the robustness and
scrambled-content controls below before any memory-specific interpretation is made.

All results remain conditional on the J-Lens not having passed the full
Table 3 validation battery.

In [47]:

# ============================================================
# ROBUSTNESS BATTERY FOR CORRECTED C2 ANALYSIS
#
# 1. Correct n=21 condition-level analysis
# 2. Leave-one-distance-out
# 3. Leave-one-filler-out
# 4. Per-needle analysis
# 5. Fixed blocked permutation test
#
# NO MODEL INFERENCE REQUIRED
# ============================================================

RNG = np.random.default_rng(42)
N_PERM = 20_000


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def find_col(df, candidates, label):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        f"Could not identify {label} column.\n"
        f"Tried: {candidates}\n"
        f"Available columns:\n{list(df.columns)}"
    )


def summarize_effect(x):
    x = np.asarray(x, dtype=float)
    n = len(x)

    mean_log = np.mean(x)
    se = np.std(x, ddof=1) / np.sqrt(n)

    tcrit = stats.t.ppf(0.975, df=n - 1)

    lo_log = mean_log - tcrit * se
    hi_log = mean_log + tcrit * se

    t_stat, p = stats.ttest_1samp(x, 0.0)

    return {
        "n": n,
        "mean_log": mean_log,
        "fold": np.exp(mean_log),
        "CI_low": np.exp(lo_log),
        "CI_high": np.exp(hi_log),
        "t": t_stat,
        "p": p,
    }


# ------------------------------------------------------------
# Detect needle column
# ------------------------------------------------------------

needle_col = find_col(
    df_corrected,
    ["needle", "stored_needle", "target_needle", "needle_word"],
    "needle"
)

print("Using needle column:", needle_col)


# ============================================================
# 1. CORRECT CONDITION-LEVEL DATA
# ============================================================

cond = (
    df_corrected
    .groupby(
        ["layer", "requested_distance", "filler_idx"],
        as_index=False
    )
    .agg(
        delta_log_ratio=("delta_log_ratio", "mean"),
        n_needles=("delta_log_ratio", "size")
    )
)

assert (cond["n_needles"] == 4).all(), \
    "Some conditions do not contain exactly 4 needles."

print("\n" + "=" * 70)
print("1. CORRECTED CONDITION-LEVEL ANALYSIS")
print("=" * 70)

print("Original rows:", len(df_corrected))
print("Condition rows:", len(cond))
print("\nConditions per layer:")
print(cond.groupby("layer").size())

baseline_rows = []

for layer, g in cond.groupby("layer"):
    r = summarize_effect(g["delta_log_ratio"])
    r["layer"] = layer
    baseline_rows.append(r)

baseline = pd.DataFrame(baseline_rows)[
    ["layer", "n", "mean_log", "fold", "CI_low", "CI_high", "t", "p"]
]

print(
    "\n" +
    baseline.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# 2. LEAVE-ONE-DISTANCE-OUT
# ============================================================

print("\n" + "=" * 70)
print("2. LEAVE-ONE-DISTANCE-OUT")
print("=" * 70)

lodo_rows = []

for layer in sorted(cond["layer"].unique()):

    layer_df = cond[
        cond["layer"] == layer
    ]

    for dropped in sorted(
        layer_df["requested_distance"].unique()
    ):

        g = layer_df[
            layer_df["requested_distance"] != dropped
        ]

        r = summarize_effect(
            g["delta_log_ratio"]
        )

        lodo_rows.append({
            "layer": layer,
            "dropped_distance": dropped,
            **r
        })

lodo = pd.DataFrame(lodo_rows)

print(
    lodo[
        [
            "layer",
            "dropped_distance",
            "n",
            "fold",
            "CI_low",
            "CI_high",
            "p"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# 3. LEAVE-ONE-FILLER-OUT
# ============================================================

print("\n" + "=" * 70)
print("3. LEAVE-ONE-FILLER-OUT")
print("=" * 70)

lofo_rows = []

for layer in sorted(cond["layer"].unique()):

    layer_df = cond[
        cond["layer"] == layer
    ]

    for dropped in sorted(
        layer_df["filler_idx"].unique()
    ):

        g = layer_df[
            layer_df["filler_idx"] != dropped
        ]

        r = summarize_effect(
            g["delta_log_ratio"]
        )

        lofo_rows.append({
            "layer": layer,
            "dropped_filler": dropped,
            **r
        })

lofo = pd.DataFrame(lofo_rows)

print(
    lofo[
        [
            "layer",
            "dropped_filler",
            "n",
            "fold",
            "CI_low",
            "CI_high",
            "p"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# 4. PER-NEEDLE ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("4. PER-NEEDLE ANALYSIS")
print("=" * 70)

needle_rows = []

for (layer, needle), g in df_corrected.groupby(
    ["layer", needle_col]
):

    r = summarize_effect(
        g["delta_log_ratio"]
    )

    needle_rows.append({
        "layer": layer,
        "needle": needle,
        **r
    })

needle_results = pd.DataFrame(
    needle_rows
)

print(
    needle_results[
        [
            "layer",
            "needle",
            "n",
            "fold",
            "CI_low",
            "CI_high",
            "p"
        ]
    ]
    .sort_values(
        ["layer", "needle"]
    )
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# 5. BLOCKED PERMUTATION TEST
# ============================================================

print("\n" + "=" * 70)
print("5. BLOCKED PERMUTATION TEST")
print("=" * 70)

required_bias = {
    "stored_needle",
    "tested_needle",
    "distractor",
    "layer",
    "requested_distance",
    "filler_idx",
    "p_needle",
    "p_distractor",
}

missing = required_bias - set(
    df_bias.columns
)

assert not missing, \
    f"Missing df_bias columns: {missing}"

bias = df_bias.copy()


# ------------------------------------------------------------
# Check probabilities before logs
# ------------------------------------------------------------

min_prob = min(
    bias["p_needle"].min(),
    bias["p_distractor"].min()
)

print("Minimum probability:", min_prob)

if min_prob <= 0:
    raise ValueError(
        "Zero/negative probability found. "
        "Cannot compute safe log-ratios."
    )


bias["raw_log_ratio"] = (
    np.log(bias["p_needle"])
    -
    np.log(bias["p_distractor"])
)


# ------------------------------------------------------------
# Each block:
# layer × distance × filler × tested pair
#
# Within each block there should be 4 stored prompts.
# ------------------------------------------------------------

block_cols = [
    "layer",
    "requested_distance",
    "filler_idx",
    "tested_needle",
    "distractor"
]

block_sizes = (
    bias
    .groupby(block_cols)
    .size()
)

print("\nBlock-size counts:")
print(
    block_sizes
    .value_counts()
    .sort_index()
)

bad_blocks = block_sizes[
    block_sizes != 4
]

if len(bad_blocks) > 0:
    print("\nBad blocks:")
    print(
        bad_blocks.head(20)
    )
    raise ValueError(
        f"{len(bad_blocks)} blocks do not "
        "contain exactly 4 stored prompts."
    )

print(
    "All matched blocks contain exactly "
    "4 stored prompts: PASS"
)


# ------------------------------------------------------------
# Build blocks
# ------------------------------------------------------------

blocks = []

for key, g in bias.groupby(
    block_cols
):

    # This ordering is important:
    # index 0-3 must correspond to the same
    # stored identities across tested pairs.
    g = g.sort_values(
        "stored_needle"
    )

    vals = (
        g["raw_log_ratio"]
        .to_numpy(dtype=float)
    )

    stored_order = (
        g["stored_needle"]
        .tolist()
    )

    blocks.append({
        "layer": key[0],
        "distance": key[1],
        "filler": key[2],
        "tested_needle": key[3],
        "distractor": key[4],
        "vals": vals,
        "stored_order": stored_order
    })


# ------------------------------------------------------------
# Verify stored ordering is identical everywhere
# ------------------------------------------------------------

all_orders = {
    tuple(b["stored_order"])
    for b in blocks
}

print(
    "\nUnique stored-needle orders:",
    all_orders
)

if len(all_orders) != 1:
    raise ValueError(
        "Stored-needle ordering is not "
        "consistent across blocks."
    )

print(
    "Stored-needle ordering consistent: PASS"
)


# ------------------------------------------------------------
# Observed corrected effect
# ------------------------------------------------------------

observed = (
    cond
    .groupby("layer")[
        "delta_log_ratio"
    ]
    .mean()
    .to_dict()
)

print(
    "\nObserved corrected effects:"
)

for layer, obs in observed.items():
    print(
        f"Layer {layer}: "
        f"mean_log={obs:.6f}, "
        f"fold={np.exp(obs):.6f}x"
    )


# ------------------------------------------------------------
# FIXED BLOCKED PERMUTATION
#
# For each distance × filler condition:
#
# randomly permute the four stored-prompt
# identities exactly once.
#
# The same one-to-one assignment is then
# applied across the four tested pairs.
#
# This avoids sampling targets with replacement.
# ------------------------------------------------------------

perm_results = []

for layer in sorted(
    cond["layer"].unique()
):

    layer_blocks = [
        b
        for b in blocks
        if b["layer"] == layer
    ]

    condition_keys = sorted({
        (
            b["distance"],
            b["filler"]
        )
        for b in layer_blocks
    })

    print(
        f"\nLayer {layer}: "
        f"{len(layer_blocks)} matched blocks, "
        f"{len(condition_keys)} conditions"
    )

    assert len(condition_keys) == 21, (
        f"Expected 21 conditions at "
        f"layer {layer}; "
        f"found {len(condition_keys)}"
    )

    null_stats = np.empty(
        N_PERM,
        dtype=float
    )

    for perm_i in range(
        N_PERM
    ):

        condition_effects = []

        for distance, filler in condition_keys:

            these_blocks = [
                b
                for b in layer_blocks
                if (
                    b["distance"] == distance
                    and
                    b["filler"] == filler
                )
            ]

            # stable order of tested pairs
            these_blocks = sorted(
                these_blocks,
                key=lambda b: (
                    b["tested_needle"],
                    b["distractor"]
                )
            )

            if len(these_blocks) != 4:
                raise ValueError(
                    f"Expected 4 tested-pair blocks "
                    f"for layer={layer}, "
                    f"distance={distance}, "
                    f"filler={filler}; "
                    f"found {len(these_blocks)}"
                )

            # ------------------------------------
            # TRUE ONE-TO-ONE PERMUTATION
            # ------------------------------------

            assignment = (
                RNG.permutation(4)
            )

            pair_effects = []

            for j, b in enumerate(
                these_blocks
            ):

                vals = b["vals"]

                idx = assignment[j]

                pseudo_target = (
                    vals[idx]
                )

                pseudo_baseline = (
                    np.delete(
                        vals,
                        idx
                    ).mean()
                )

                pseudo_delta = (
                    pseudo_target
                    -
                    pseudo_baseline
                )

                pair_effects.append(
                    pseudo_delta
                )

            # Average four tested-pair effects
            # inside this distance × filler condition
            condition_effects.append(
                np.mean(pair_effects)
            )

        # same statistic as corrected n=21 analysis
        null_stats[perm_i] = (
            np.mean(
                condition_effects
            )
        )

    obs = observed[layer]

    p_perm = (
        np.sum(
            np.abs(null_stats)
            >=
            abs(obs)
        )
        + 1
    ) / (
        N_PERM + 1
    )

    perm_results.append({
        "layer": layer,
        "observed_mean_log": obs,
        "observed_fold": np.exp(obs),
        "null_mean": np.mean(null_stats),
        "null_sd": np.std(
            null_stats,
            ddof=1
        ),
        "permutation_p": p_perm
    })


perm_results = pd.DataFrame(
    perm_results
)

print(
    "\n" + "=" * 70
)

print(
    "BLOCKED PERMUTATION RESULTS"
)

print(
    "=" * 70
)

print(
    perm_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# FINAL ROBUSTNESS SUMMARY
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "FINAL ROBUSTNESS SUMMARY"
)

print(
    "=" * 70
)


for layer in sorted(
    cond["layer"].unique()
):

    base = baseline[
        baseline["layer"] == layer
    ].iloc[0]

    ld = lodo[
        lodo["layer"] == layer
    ]

    lf = lofo[
        lofo["layer"] == layer
    ]

    nd = needle_results[
        needle_results["layer"] == layer
    ]

    pp = perm_results[
        perm_results["layer"] == layer
    ].iloc[0]

    print(
        f"\nLAYER {layer}"
    )

    print(
        f"  Main corrected fold:    "
        f"{base['fold']:.4f}x"
    )

    print(
        f"  Main 95% CI:            "
        f"[{base['CI_low']:.4f}, "
        f"{base['CI_high']:.4f}]"
    )

    print(
        f"  Main t-test p:          "
        f"{base['p']:.6g}"
    )

    print(
        f"  Leave-distance folds:   "
        f"{ld['fold'].min():.4f}x "
        f"to "
        f"{ld['fold'].max():.4f}x"
    )

    print(
        f"  Leave-filler folds:     "
        f"{lf['fold'].min():.4f}x "
        f"to "
        f"{lf['fold'].max():.4f}x"
    )

    print(
        f"  Per-needle folds:       "
        f"{nd['fold'].min():.4f}x "
        f"to "
        f"{nd['fold'].max():.4f}x"
    )

    print(
        f"  Block permutation p:    "
        f"{pp['permutation_p']:.6g}"
    )


print(
    "\nDONE."
)

print(
    "Send the full FINAL ROBUSTNESS SUMMARY "
    "and BLOCKED PERMUTATION RESULTS for interpretation."
)

Using needle column: needle

1. CORRECTED CONDITION-LEVEL ANALYSIS
Original rows: 252
Condition rows: 63

Conditions per layer:
layer
9     21
18    21
27    21
dtype: int64

 layer  n    mean_log     fold   CI_low  CI_high         t          p
     9 21 -0.00479086 0.995221 0.983423  1.00716 -0.838024   0.411921
    18 21   0.0456811  1.04674  1.02062  1.07353   3.77042 0.00120271
    27 21 -0.00879179 0.991247 0.962098  1.02128 -0.614446   0.545848

2. LEAVE-ONE-DISTANCE-OUT
 layer  dropped_distance  n     fold   CI_low  CI_high           p
     9                64 18  1.00148 0.990571   1.0125    0.779538
     9               256 18 0.994945 0.982279  1.00777     0.41559
     9               512 18 0.993499 0.980473   1.0067    0.311721
     9              1024 18 0.991532 0.979121   1.0041    0.172443
     9              2048 18 0.994616 0.981139  1.00828    0.415358
     9              4096 18 0.994683 0.981275  1.00827    0.418733
     9              8192 18 0.995819 0.981978  1.

In [32]:


# ============================================================
# C2 EXPANDED LEAKAGE / PROMPT-CONFOUND CHECKS
#
# Claude review follow-up #5
#
# Checks:
#   A. actual eviction distance by stored needle
#   B. requested distance balance
#   C. filler balance
#   D. needle position (if available)
#   E. prompt token length (if available)
#   F. local prompt context (if available)
#
# CPU ONLY — NO MODEL INFERENCE
# ============================================================

print("=" * 72)
print("C2 EXPANDED LEAKAGE / PROMPT-CONFOUND CHECKS")
print("=" * 72)


# ------------------------------------------------------------
# 0. Inspect what we actually have
# ------------------------------------------------------------

print("\ndf_bias shape:", df_bias.shape)
print("\ndf_bias columns:")
print(list(df_bias.columns))

required = {
    "stored_needle",
    "layer",
    "requested_distance",
    "actual_eviction_distance",
    "filler_idx",
}

missing = required - set(df_bias.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

# One prompt is repeated across tested pairs/layers.
# Reduce to unique prompt-level observations.

prompt_key_candidates = [
    "stored_needle",
    "requested_distance",
    "actual_eviction_distance",
    "filler_idx",
]

for optional in [
    "needle_pos",
    "n_tokens",
    "prompt_tokens",
    "prompt",
]:
    if optional in df_bias.columns:
        prompt_key_candidates.append(optional)

prompts = (
    df_bias[prompt_key_candidates]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nUnique prompt-level rows:", len(prompts))


# ============================================================
# A. REQUESTED DISTANCE BALANCE
# ============================================================

print("\n" + "=" * 72)
print("A. REQUESTED DISTANCE × STORED NEEDLE")
print("=" * 72)

requested_table = pd.crosstab(
    prompts["requested_distance"],
    prompts["stored_needle"]
)

print(requested_table)

balanced_requested = (
    requested_table.nunique(axis=1) == 1
).all()

print(
    "\nBalanced across stored needles:",
    "PASS" if balanced_requested else "CHECK"
)


# ============================================================
# B. FILLER BALANCE
# ============================================================

print("\n" + "=" * 72)
print("B. FILLER × STORED NEEDLE")
print("=" * 72)

filler_table = pd.crosstab(
    prompts["filler_idx"],
    prompts["stored_needle"]
)

print(filler_table)

balanced_filler = (
    filler_table.nunique(axis=1) == 1
).all()

print(
    "\nBalanced across stored needles:",
    "PASS" if balanced_filler else "CHECK"
)


# ============================================================
# C. ACTUAL EVICTION DISTANCE
# ============================================================

print("\n" + "=" * 72)
print("C. ACTUAL EVICTION DISTANCE BY STORED NEEDLE")
print("=" * 72)

eviction_summary = (
    prompts
    .groupby("stored_needle")[
        "actual_eviction_distance"
    ]
    .agg(
        n="count",
        mean="mean",
        std="std",
        min="min",
        median="median",
        max="max"
    )
)

print(eviction_summary)

print("\nMean actual eviction distance by requested distance:")

eviction_by_requested = (
    prompts
    .groupby(
        ["requested_distance", "stored_needle"]
    )["actual_eviction_distance"]
    .mean()
    .unstack()
)

print(eviction_by_requested)

# Range across needles within each requested distance
eviction_spread = (
    eviction_by_requested.max(axis=1)
    -
    eviction_by_requested.min(axis=1)
)

print("\nMax needle-to-needle spread within each requested distance:")
print(eviction_spread)

print(
    "\nLargest spread:",
    eviction_spread.max()
)


# ============================================================
# D. NEEDLE POSITION
# ============================================================

print("\n" + "=" * 72)
print("D. NEEDLE POSITION")
print("=" * 72)

if "needle_pos" in df_bias.columns:

    pos_prompts = (
        df_bias[
            [
                "stored_needle",
                "requested_distance",
                "filler_idx",
                "needle_pos"
            ]
        ]
        .drop_duplicates()
    )

    pos_summary = (
        pos_prompts
        .groupby("stored_needle")[
            "needle_pos"
        ]
        .agg(
            n="count",
            mean="mean",
            std="std",
            min="min",
            median="median",
            max="max"
        )
    )

    print(pos_summary)

    pos_by_condition = (
        pos_prompts
        .pivot_table(
            index=[
                "requested_distance",
                "filler_idx"
            ],
            columns="stored_needle",
            values="needle_pos",
            aggfunc="mean"
        )
    )

    pos_spread = (
        pos_by_condition.max(axis=1)
        -
        pos_by_condition.min(axis=1)
    )

    print(
        "\nLargest within-condition "
        "needle-position spread:",
        pos_spread.max()
    )

else:
    print(
        "NOT AVAILABLE in df_bias.\n"
        "Cannot claim needle-position leakage has been ruled out "
        "from this dataframe."
    )


# ============================================================
# E. PROMPT TOKEN LENGTH
# ============================================================

print("\n" + "=" * 72)
print("E. PROMPT TOKEN LENGTH")
print("=" * 72)

token_length_col = None

for candidate in [
    "n_tokens",
    "prompt_n_tokens",
    "prompt_length",
    "token_length"
]:
    if candidate in df_bias.columns:
        token_length_col = candidate
        break

if token_length_col is not None:

    len_prompts = (
        df_bias[
            [
                "stored_needle",
                "requested_distance",
                "filler_idx",
                token_length_col
            ]
        ]
        .drop_duplicates()
    )

    length_summary = (
        len_prompts
        .groupby("stored_needle")[
            token_length_col
        ]
        .agg(
            n="count",
            mean="mean",
            std="std",
            min="min",
            median="median",
            max="max"
        )
    )

    print(length_summary)

    length_by_condition = (
        len_prompts
        .pivot_table(
            index=[
                "requested_distance",
                "filler_idx"
            ],
            columns="stored_needle",
            values=token_length_col,
            aggfunc="mean"
        )
    )

    length_spread = (
        length_by_condition.max(axis=1)
        -
        length_by_condition.min(axis=1)
    )

    print(
        "\nLargest within-condition "
        "token-length spread:",
        length_spread.max()
    )

else:
    print(
        "NOT AVAILABLE in df_bias.\n"
        "Cannot claim prompt-token-length leakage has been "
        "ruled out from this dataframe."
    )


# ============================================================
# F. LOCAL CONTEXT
# ============================================================

print("\n" + "=" * 72)
print("F. LOCAL CONTEXT AROUND NEEDLE")
print("=" * 72)

context_cols = [
    c for c in [
        "prompt",
        "local_context",
        "needle_context",
        "context"
    ]
    if c in df_bias.columns
]

if context_cols:

    print("Available context columns:", context_cols)

    for c in context_cols:
        print(
            f"\nUnique {c} values:",
            df_bias[c].nunique()
        )

    print(
        "\nContext text is available. "
        "Inspect matched windows around the needle before "
        "declaring this check passed."
    )

else:
    print(
        "NOT AVAILABLE in df_bias.\n"
        "Local filler/context equivalence cannot be verified "
        "from this dataframe."
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 72)
print("LEAKAGE CHECK STATUS")
print("=" * 72)

print(
    "Requested-distance balance:",
    "PASS" if balanced_requested else "CHECK"
)

print(
    "Filler balance:",
    "PASS" if balanced_filler else "CHECK"
)

print(
    "Actual eviction distance:",
    "INSPECT TABLE ABOVE"
)

print(
    "Needle position:",
    "AVAILABLE" if "needle_pos" in df_bias.columns
    else "NOT AVAILABLE"
)

print(
    "Prompt token length:",
    "AVAILABLE" if token_length_col is not None
    else "NOT AVAILABLE"
)

print(
    "Local prompt context:",
    "AVAILABLE" if context_cols
    else "NOT AVAILABLE"
)

print("\nDONE.")

C2 EXPANDED LEAKAGE / PROMPT-CONFOUND CHECKS

df_bias shape: (1008, 10)

df_bias columns:
['stored_needle', 'tested_needle', 'distractor', 'layer', 'requested_distance', 'actual_eviction_distance', 'filler_idx', 'p_needle', 'p_distractor', 'ratio']

Unique prompt-level rows: 84

A. REQUESTED DISTANCE × STORED NEEDLE
stored_needle       Paris  Tokyo  banana  lantern
requested_distance                               
64                      3      3       3        3
256                     3      3       3        3
512                     3      3       3        3
1024                    3      3       3        3
2048                    3      3       3        3
4096                    3      3       3        3
8192                    3      3       3        3

Balanced across stored needles: PASS

B. FILLER × STORED NEEDLE
stored_needle  Paris  Tokyo  banana  lantern
filler_idx                                  
0                  7      7       7        7
1                  7      7     

In [33]:
# ============================================================
# C2 TOKEN LENGTH + NEEDLE POSITION CHECK
# CPU ONLY
# ============================================================


print("=" * 72)
print("C2 TOKEN-LENGTH / NEEDLE-POSITION CHECK")
print("=" * 72)

C2_NEEDLES = ["Paris", "Tokyo", "banana", "lantern"]

# Use c2 because it still contains the original metadata
meta = (
    c2[
        c2["needle"].isin(C2_NEEDLES)
    ][
        [
            "needle",
            "requested_distance",
            "filler_idx",
            "n_tokens",
            "needle_pos",
            "eviction_distance",
        ]
    ]
    .drop_duplicates()
    .copy()
)

print("\nUnique metadata rows:", len(meta))

print("\nRows per needle:")
print(meta.groupby("needle").size())


# ============================================================
# 1. TOKEN LENGTH
# ============================================================

print("\n" + "=" * 72)
print("1. TOKEN LENGTH BY NEEDLE")
print("=" * 72)

print(
    meta.groupby("needle")["n_tokens"]
    .agg(["count", "mean", "std", "min", "median", "max"])
)


# ============================================================
# 2. NEEDLE POSITION
# ============================================================

print("\n" + "=" * 72)
print("2. NEEDLE POSITION BY NEEDLE")
print("=" * 72)

print(
    meta.groupby("needle")["needle_pos"]
    .agg(["count", "mean", "std", "min", "median", "max"])
)


# ============================================================
# 3. WITHIN-CONDITION SPREADS
# ============================================================

length_pivot = meta.pivot_table(
    index=["requested_distance", "filler_idx"],
    columns="needle",
    values="n_tokens",
    aggfunc="mean"
)

pos_pivot = meta.pivot_table(
    index=["requested_distance", "filler_idx"],
    columns="needle",
    values="needle_pos",
    aggfunc="mean"
)

evict_pivot = meta.pivot_table(
    index=["requested_distance", "filler_idx"],
    columns="needle",
    values="eviction_distance",
    aggfunc="mean"
)

length_spread = length_pivot.max(axis=1) - length_pivot.min(axis=1)
pos_spread = pos_pivot.max(axis=1) - pos_pivot.min(axis=1)
evict_spread = evict_pivot.max(axis=1) - evict_pivot.min(axis=1)


# ============================================================
# 4. RESULTS
# ============================================================

print("\n" + "=" * 72)
print("WITHIN-CONDITION SPREADS")
print("=" * 72)

print("\nToken-length spread:")
print(length_spread)

print("\nNeedle-position spread:")
print(pos_spread)

print("\nEviction-distance spread:")
print(evict_spread)


# ============================================================
# FINAL
# ============================================================

length_pass = (length_spread == 0).all()
position_pass = (pos_spread == 0).all()
eviction_pass = (evict_spread == 0).all()

print("\n" + "=" * 72)
print("FINAL METADATA CHECK")
print("=" * 72)

print(
    f"Token length:    "
    f"{'PASS' if length_pass else 'CHECK'} "
    f"(max spread={length_spread.max()})"
)

print(
    f"Needle position: "
    f"{'PASS' if position_pass else 'CHECK'} "
    f"(max spread={pos_spread.max()})"
)

print(
    f"Eviction dist.:  "
    f"{'PASS' if eviction_pass else 'CHECK'} "
    f"(max spread={evict_spread.max()})"
)

print("DONE")

C2 TOKEN-LENGTH / NEEDLE-POSITION CHECK

Unique metadata rows: 96

Rows per needle:
needle
Paris      24
Tokyo      24
banana     24
lantern    24
dtype: int64

1. TOKEN LENGTH BY NEEDLE
         count          mean          std   min  median    max
needle                                                        
Paris       24  10286.416667  2696.133156  8292  9001.0  16434
Tokyo       24  10286.416667  2696.133156  8292  9001.0  16434
banana      24  10286.416667  2696.133156  8292  9001.0  16434
lantern     24  10286.416667  2696.133156  8292  9001.0  16434

2. NEEDLE POSITION BY NEEDLE
         count    mean          std  min  median   max
needle                                                
Paris       24  1185.0  2806.262996  141   145.0  8461
Tokyo       24  1185.0  2806.262996  141   145.0  8461
banana      24  1185.0  2806.262996  141   145.0  8461
lantern     24  1185.0  2806.262996  141   145.0  8461

WITHIN-CONDITION SPREADS

Token-length spread:
requested_distance  filler_

In [34]:
# ============================================================
# C2 #6 — SCRAMBLED-NEEDLE CONTENT CONTROL
#
# Directly mirrors Cell 41:
#   same build_niah_prompt()
#   same distances
#   same fillers
#   same probe.run()
#   same layers
#   same J-Lens readout
#
# For each C2 needle, replace the stored word with a different
# single-token content word, while still testing the ORIGINAL
# needle/distractor pair.
# ============================================================

LAYERS = [9, 18, 27]
DISTANCES = EXP["eviction_distances"]
FILLERS = range(EXP["n_filler_variants"])

PAIRS = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

# Candidate replacement words.
# We verify them with the SAME tokenization convention as Cell 41:
# tok.encode(f" {word}", add_special_tokens=False)
CONTROL_POOL = [
    "river", "chair", "window", "garden",
    "table", "house", "water", "paper",
    "stone", "music", "green", "cloud",
    "flower", "coffee", "bridge", "forest",
    "silver", "camera", "bottle", "pencil",
    "yellow", "summer", "winter", "kitchen",
    "street", "book", "door", "tree",
]

# ============================================================
# 1. VERIFY ORIGINAL PAIRS + CHOOSE VALID CONTROL WORDS
# ============================================================

pair_ids = {}

for needle, distractor in PAIRS.items():

    n_ids = tok.encode(
        f" {needle}",
        add_special_tokens=False,
    )

    d_ids = tok.encode(
        f" {distractor}",
        add_special_tokens=False,
    )

    assert len(n_ids) == 1, (
        f"{needle} is not single-token: {n_ids}"
    )

    assert len(d_ids) == 1, (
        f"{distractor} is not single-token: {d_ids}"
    )

    pair_ids[needle] = (
        n_ids[0],
        d_ids[0],
    )


# Only retain genuinely single-token controls under the
# exact tokenization convention used by Cell 41.

forbidden_words = (
    set(PAIRS.keys())
    | set(PAIRS.values())
)

valid_controls = []

for word in CONTROL_POOL:

    ids = tok.encode(
        f" {word}",
        add_special_tokens=False,
    )

    if (
        len(ids) == 1
        and word not in forbidden_words
    ):
        valid_controls.append(word)


assert len(valid_controls) >= len(PAIRS), (
    "Not enough verified single-token control words."
)


# Deterministic assignment so the experiment is reproducible.
CONTROL_MAP = {
    needle: valid_controls[i]
    for i, needle in enumerate(PAIRS)
}


print("=" * 72)
print("SCRAMBLED-NEEDLE CONTROL MAP")
print("=" * 72)

for needle, control in CONTROL_MAP.items():

    original_ids = tok.encode(
        f" {needle}",
        add_special_tokens=False,
    )

    control_ids = tok.encode(
        f" {control}",
        add_special_tokens=False,
    )

    print(
        f"{needle:8s} -> {control:8s} | "
        f"{original_ids} -> {control_ids}"
    )

    assert len(original_ids) == 1
    assert len(control_ids) == 1
    assert original_ids[0] != control_ids[0]


# ============================================================
# 2. VERIFY df_bias BEFORE USING IT AS ORIGINAL TARGET
# ============================================================

required_cols = {
    "stored_needle",
    "tested_needle",
    "distractor",
    "layer",
    "requested_distance",
    "actual_eviction_distance",
    "filler_idx",
    "p_needle",
    "p_distractor",
}

missing = required_cols - set(df_bias.columns)

assert not missing, (
    f"df_bias missing columns: {missing}"
)

original_target = df_bias[
    df_bias["stored_needle"]
    == df_bias["tested_needle"]
].copy()

# Expected:
# 4 needles × 7 distances × 3 fillers × 3 layers = 252
assert len(original_target) == 252, (
    f"Expected 252 original target rows, "
    f"found {len(original_target)}"
)

assert (
    original_target.groupby(
        [
            "stored_needle",
            "layer",
            "requested_distance",
            "filler_idx",
        ]
    ).size() == 1
).all()


# ============================================================
# 3. PRE-FLIGHT STRUCTURAL CHECK
#
# Build prompts only first.
# Verify replacing the needle does NOT alter:
#   - prompt token count
#   - needle position
#   - actual eviction distance
#   - AHN activation
#   - eviction status
#
# No probe.run() happens until ALL conditions pass.
# ============================================================

preflight_rows = []

for original_needle, control_needle in CONTROL_MAP.items():

    for requested_distance in DISTANCES:

        for filler_idx in FILLERS:

            original_spec = ai.build_niah_prompt(
                tok,
                original_needle,
                bundle,
                eviction_distance=requested_distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            control_spec = ai.build_niah_prompt(
                tok,
                control_needle,
                bundle,
                eviction_distance=requested_distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            preflight_rows.append({
                "original_needle": original_needle,
                "control_needle": control_needle,
                "requested_distance": requested_distance,
                "filler_idx": filler_idx,

                "original_n_tokens":
                    original_spec["n_tokens"],

                "control_n_tokens":
                    control_spec["n_tokens"],

                "original_needle_pos":
                    original_spec["needle_pos"],

                "control_needle_pos":
                    control_spec["needle_pos"],

                "original_eviction_distance":
                    original_spec[
                        "actual_eviction_distance"
                    ],

                "control_eviction_distance":
                    control_spec[
                        "actual_eviction_distance"
                    ],

                "original_ahn_active":
                    original_spec["ahn_will_activate"],

                "control_ahn_active":
                    control_spec["ahn_will_activate"],

                "original_evicted":
                    original_spec["needle_is_evicted"],

                "control_evicted":
                    control_spec["needle_is_evicted"],
            })


df_content_preflight = pd.DataFrame(
    preflight_rows
)

# Same 84 conditions as Cell 41.
assert len(df_content_preflight) == 84


df_content_preflight["token_diff"] = (
    df_content_preflight["control_n_tokens"]
    - df_content_preflight["original_n_tokens"]
)

df_content_preflight["position_diff"] = (
    df_content_preflight["control_needle_pos"]
    - df_content_preflight["original_needle_pos"]
)

df_content_preflight["eviction_diff"] = (
    df_content_preflight["control_eviction_distance"]
    - df_content_preflight["original_eviction_distance"]
)


print("\n" + "=" * 72)
print("STRUCTURAL PREFLIGHT")
print("=" * 72)

print(
    "Max |token-count difference|:",
    df_content_preflight[
        "token_diff"
    ].abs().max()
)

print(
    "Max |needle-position difference|:",
    df_content_preflight[
        "position_diff"
    ].abs().max()
)

print(
    "Max |eviction-distance difference|:",
    df_content_preflight[
        "eviction_diff"
    ].abs().max()
)


assert (
    df_content_preflight["token_diff"] == 0
).all(), "STOP: token counts differ."

assert (
    df_content_preflight["position_diff"] == 0
).all(), "STOP: needle positions differ."

assert (
    df_content_preflight["eviction_diff"] == 0
).all(), "STOP: eviction distances differ."

assert (
    df_content_preflight[
        "original_ahn_active"
    ]
    ==
    df_content_preflight[
        "control_ahn_active"
    ]
).all(), "STOP: AHN activation differs."

assert (
    df_content_preflight[
        "original_evicted"
    ]
    ==
    df_content_preflight[
        "control_evicted"
    ]
).all(), "STOP: eviction status differs."

assert (
    df_content_preflight[
        "control_ahn_active"
    ]
).all(), "STOP: a control prompt does not activate AHN."

assert (
    df_content_preflight[
        "control_evicted"
    ]
).all(), "STOP: a control needle is not evicted."


print("Structural preflight: PASS")


# ============================================================
# 4. CONTROL SWEEP
#
# Critical point:
# control_needle is what is STORED,
# but we read out the ORIGINAL needle/distractor pair.
#
# Example:
#   stored = river
#   tested = Paris vs London
#
# This directly tests Claude's content-conditional drift concern.
# ============================================================

c2_content_control_rows = []

total = (
    len(CONTROL_MAP)
    * len(DISTANCES)
    * len(list(FILLERS))
)

done = 0

for original_needle, control_needle in CONTROL_MAP.items():

    needle_id, distractor_id = pair_ids[
        original_needle
    ]

    distractor = PAIRS[
        original_needle
    ]

    for requested_distance in DISTANCES:

        for filler_idx in FILLERS:

            spec = ai.build_niah_prompt(
                tok,
                control_needle,
                bundle,
                eviction_distance=requested_distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            assert spec["ahn_will_activate"]
            assert spec["needle_is_evicted"]

            ins = tok(
                spec["prompt"],
                return_tensors="pt",
            ).to(bundle.model.device)

            on = probe.run(
                ins,
                nowrite=False,
                layers=LAYERS,
                capture_residual=False,
            )

            for L in LAYERS:

                if L not in on.ahn_raw:
                    continue

                o_t = on.o_t(
                    L,
                    pos=-1,
                )

                logits = ai.readout_logits(
                    o_t,
                    bundle,
                    lens=lens,
                    layer=L,
                )

                p_n = ai.token_prob(
                    logits,
                    needle_id,
                )

                p_d = ai.token_prob(
                    logits,
                    distractor_id,
                )

                assert p_n > 0
                assert p_d > 0

                c2_content_control_rows.append({
                    "original_needle":
                        original_needle,

                    "control_needle":
                        control_needle,

                    "tested_needle":
                        original_needle,

                    "distractor":
                        distractor,

                    "layer":
                        L,

                    "requested_distance":
                        requested_distance,

                    "actual_eviction_distance":
                        spec[
                            "actual_eviction_distance"
                        ],

                    "filler_idx":
                        filler_idx,

                    "n_tokens":
                        spec["n_tokens"],

                    "needle_pos":
                        spec["needle_pos"],

                    "p_needle":
                        p_n,

                    "p_distractor":
                        p_d,

                    "log_ratio":
                        np.log(p_n)
                        - np.log(p_d),
                })

            done += 1

            if (
                done % 10 == 0
                or done == total
            ):
                print(
                    f"{done}/{total} "
                    "forward passes completed"
                )


df_content_control = pd.DataFrame(
    c2_content_control_rows
)


# ============================================================
# 5. POST-RUN INTEGRITY CHECKS
# ============================================================

assert len(df_content_control) == 252, (
    f"Expected 252 control rows, "
    f"found {len(df_content_control)}"
)

assert (
    df_content_control.groupby(
        [
            "original_needle",
            "layer",
            "requested_distance",
            "filler_idx",
        ]
    ).size() == 1
).all()


# Original target log-ratio.
# Do NOT use EPS here: probabilities were already produced and
# should be strictly positive. Fail loudly if they are not.

assert (
    original_target["p_needle"] > 0
).all()

assert (
    original_target["p_distractor"] > 0
).all()

original_target[
    "original_log_ratio"
] = (
    np.log(
        original_target["p_needle"]
    )
    -
    np.log(
        original_target["p_distractor"]
    )
)


original_for_merge = (
    original_target[
        [
            "stored_needle",
            "layer",
            "requested_distance",
            "filler_idx",
            "actual_eviction_distance",
            "original_log_ratio",
        ]
    ]
    .rename(
        columns={
            "stored_needle":
                "original_needle",

            "actual_eviction_distance":
                "original_eviction_distance",
        }
    )
)


paired_content = original_for_merge.merge(
    df_content_control[
        [
            "original_needle",
            "control_needle",
            "layer",
            "requested_distance",
            "filler_idx",
            "actual_eviction_distance",
            "log_ratio",
        ]
    ].rename(
        columns={
            "actual_eviction_distance":
                "control_eviction_distance",

            "log_ratio":
                "control_log_ratio",
        }
    ),

    on=[
        "original_needle",
        "layer",
        "requested_distance",
        "filler_idx",
    ],

    how="inner",
    validate="one_to_one",
)


assert len(paired_content) == 252


assert (
    paired_content[
        "original_eviction_distance"
    ]
    ==
    paired_content[
        "control_eviction_distance"
    ]
).all()


# ============================================================
# 6. ORIGINAL vs SCRAMBLED-CONTENT EFFECT
#
# Positive:
# original stored word raises its own needle/distractor
# readout relative to the neutral replacement.
#
# Zero:
# original and replacement content behave the same.
#
# Negative:
# original stored word lowers its own pair readout.
# ============================================================

paired_content[
    "delta_log_original_vs_control"
] = (
    paired_content[
        "original_log_ratio"
    ]
    -
    paired_content[
        "control_log_ratio"
    ]
)

paired_content[
    "fold_original_vs_control"
] = np.exp(
    paired_content[
        "delta_log_original_vs_control"
    ]
)


# ============================================================
# 7. CONDITION-LEVEL SUMMARY
#
# Same unit used in corrected analysis:
# average four needle/control comparisons inside each
# layer × distance × filler condition.
# ============================================================

content_cond = (
    paired_content
    .groupby(
        [
            "layer",
            "requested_distance",
            "filler_idx",
        ],
        as_index=False,
    )
    .agg(
        delta_log=(
            "delta_log_original_vs_control",
            "mean",
        ),

        n_pairs=(
            "delta_log_original_vs_control",
            "size",
        ),
    )
)


assert (
    content_cond["n_pairs"] == 4
).all()

assert len(content_cond) == 63


# ============================================================
# 8. REPORT
# ============================================================

print("\n" + "=" * 72)
print("C2 #6 SCRAMBLED-NEEDLE CONTENT CONTROL")
print("=" * 72)

print(
    "\nControl rows:",
    len(df_content_control),
)

print(
    "Paired rows:",
    len(paired_content),
)

print(
    "Condition-level rows:",
    len(content_cond),
)


content_summary_rows = []

for L, g in content_cond.groupby("layer"):

    x = g["delta_log"].to_numpy(
        dtype=float
    )

    n = len(x)

    assert n == 21

    mean_log = x.mean()

    se = (
        x.std(ddof=1)
        / np.sqrt(n)
    )

    tcrit = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_log_low = (
        mean_log
        - tcrit * se
    )

    ci_log_high = (
        mean_log
        + tcrit * se
    )

    t_stat, p_value = (
        stats.ttest_1samp(
            x,
            popmean=0.0,
        )
    )

    content_summary_rows.append({
        "layer":
            L,

        "n_conditions":
            n,

        "mean_log_effect":
            mean_log,

        "geom_fold":
            np.exp(mean_log),

        "CI_low":
            np.exp(ci_log_low),

        "CI_high":
            np.exp(ci_log_high),

        "t":
            t_stat,

        "p":
            p_value,
    })


content_summary = pd.DataFrame(
    content_summary_rows
)


print("\n")
print(
    content_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}",
    )
)


# ============================================================
# 9. PER-NEEDLE RESULT
# ============================================================

print("\n" + "=" * 72)
print("PER-NEEDLE CONTENT CONTROL")
print("=" * 72)

per_needle_content = (
    paired_content
    .groupby(
        [
            "layer",
            "original_needle",
            "control_needle",
        ]
    )
    .agg(
        n=(
            "delta_log_original_vs_control",
            "size",
        ),

        mean_log=(
            "delta_log_original_vs_control",
            "mean",
        ),
    )
    .reset_index()
)

per_needle_content[
    "geom_fold"
] = np.exp(
    per_needle_content[
        "mean_log"
    ]
)

print(
    per_needle_content.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}",
    )
)


print("\nDONE.")

SCRAMBLED-NEEDLE CONTROL MAP
Paris    -> river    | [12095] -> [14796]
Tokyo    -> chair    | [26194] -> [10496]
banana   -> window   | [43096] -> [3241]
lantern  -> garden   | [73165] -> [13551]

STRUCTURAL PREFLIGHT
Max |token-count difference|: 0
Max |needle-position difference|: 0
Max |eviction-distance difference|: 0
Structural preflight: PASS
10/84 forward passes completed
20/84 forward passes completed
30/84 forward passes completed
40/84 forward passes completed
50/84 forward passes completed
60/84 forward passes completed
70/84 forward passes completed
80/84 forward passes completed
84/84 forward passes completed

C2 #6 SCRAMBLED-NEEDLE CONTENT CONTROL

Control rows: 252
Paired rows: 252
Condition-level rows: 63


 layer  n_conditions  mean_log_effect  geom_fold   CI_low  CI_high         t          p
     9            21        -0.040227   0.960571 0.937144 0.984584  -3.39851 0.00285112
    18            21       0.00909961    1.00914 0.974428  1.04509  0.542262   0.593629
   

# 04 — NIAH C2 Follow-up: Final Summary

**Checkpoint:** Qwen2.5-3B-Instruct + AHN-GDN  
**Readout:** J-Lens (Table 3 not passed — all findings conditional)  
**Layers:** 9, 18, 27

---

## Main result

C2 does not provide convincing memory-specific evidence on this checkpoint.

The original C2 >=10× control fails, and raw needle/distractor ratios are strongly affected by pair-specific readout preferences.

After correcting for pair-specific baseline preference and using 21 condition-level observations per layer:

| Layer | Fold | 95% CI | p |
|---:|---:|:---:|---:|
| 9  | 0.995× | [0.983, 1.007] | 0.412 |
| 18 | 1.047× | [1.021, 1.074] | 0.00120 |
| 27 | 0.991× | [0.962, 1.021] | 0.546 |

Layers 9 and 27 show no statistically reliable aggregate corrected effect.

Layer 18 shows a small positive association in the pair-baseline-corrected diagnostic.

However, the scrambled-needle content control does not preserve that Layer-18 effect:

| Layer | Fold | 95% CI | p |
|---:|---:|:---:|---:|
| 9  | 0.961× | [0.937, 0.985] | 0.00285 |
| 18 | 1.009× | [0.974, 1.045] | 0.594 |
| 27 | 0.990× | [0.942, 1.041] | 0.689 |

Therefore, the Layer-18 positive association cannot currently be treated as a memory-specific effect.

## Conclusion

- C2 fails its original >=10× criterion.
- Layer 9 shows no positive aggregate C2 effect.
- Layer 18 shows a small positive corrected association, but it does not survive the scrambled-content control.
- Layer 27 shows no aggregate corrected effect with the actual J-Lens loaded.
- No tested layer currently provides convincing positive, retention-consistent, memory-specific evidence.

This does not imply that AHN retains no information. It only means that the current C2 analysis with the J-Lens readout does not provide convincing evidence for a token-level, memory-specific retention signal on this checkpoint.

All conclusions remain conditional because the J-Lens has not passed the full Table 3 validation battery.

---

# C3 — Shuffled-Context Control (J-Lens Rerun)

The C2 investigation above is complete.

The following section begins the C3 control analysis using the actual
J-Lens readout.

C3 tests whether disrupting context order changes the retained needle
signal. Ordered and shuffled conditions will be compared using matched
experimental settings before any interpretation is made.

All C3 conclusions remain conditional on the J-Lens not having passed
the full Table 3 validation battery.

---

In [35]:
# C3 structural preflight — NO model forward passes

assert EXP["use_jlens"] is True
assert lens is not None

c3_preflight = []

for needle, needle_id in needles.items():
    for dist in (1024, 4096):
        spec = ai.build_niah_prompt(
            tok,
            needle,
            bundle,
            eviction_distance=dist,
            in_window=False,
            filler_idx=0,
        )

        ordered_prompt = spec["prompt"]

        words = ordered_prompt.split()
        np.random.default_rng(ai.SEED).shuffle(words)
        shuffled_prompt = " ".join(words)

        ordered_ids = tok(ordered_prompt, add_special_tokens=False)["input_ids"]
        shuffled_ids = tok(shuffled_prompt, add_special_tokens=False)["input_ids"]

        needle_tokens = tok.encode(f" {needle}", add_special_tokens=False)

        def find_subseq(seq, sub):
            return [
                i for i in range(len(seq) - len(sub) + 1)
                if seq[i:i + len(sub)] == sub
            ]

        ordered_pos = find_subseq(ordered_ids, needle_tokens)
        shuffled_pos = find_subseq(shuffled_ids, needle_tokens)

        c3_preflight.append({
            "needle": needle,
            "distance": dist,
            "ordered_tokens": len(ordered_ids),
            "shuffled_tokens": len(shuffled_ids),
            "token_count_delta": len(shuffled_ids) - len(ordered_ids),
            "ordered_needle_positions": ordered_pos,
            "shuffled_needle_positions": shuffled_pos,
        })

pd.DataFrame(c3_preflight)

,needle,distance,ordered_tokens,shuffled_tokens,token_count_delta,ordered_needle_positions,shuffled_needle_positions
0,Paris,1024,9252,9252,0,[148],[8940]
1,Paris,4096,12324,12324,0,[148],[12097]
2,banana,1024,9252,9252,0,[148],[8940]
3,banana,4096,12324,12324,0,[148],[12097]
4,Tokyo,1024,9252,9252,0,[148],[8940]
5,Tokyo,4096,12324,12324,0,[148],[12097]
6,violin,1024,9252,9252,0,[148],[8940]
7,violin,4096,12324,12324,0,[148],[12097]
8,cinnamon,1024,9252,9252,0,[148],[8940]
9,cinnamon,4096,12324,12324,0,[148],[12097]


In [36]:
# C3 fixed-position shuffle — structural verification only

def make_c3_fixed_shuffle(needle, distance, filler_idx=0):
    spec = ai.build_niah_prompt(
        tok,
        needle,
        bundle,
        eviction_distance=distance,
        in_window=False,
        filler_idx=filler_idx,
    )

    ordered_ids = tok(
        spec["prompt"],
        add_special_tokens=False,
    )["input_ids"]

    needle_ids = tok.encode(
        f" {needle}",
        add_special_tokens=False,
    )

    # Locate the needle exactly.
    matches = [
        i
        for i in range(len(ordered_ids) - len(needle_ids) + 1)
        if ordered_ids[i:i + len(needle_ids)] == needle_ids
    ]

    assert len(matches) == 1, (
        f"{needle}: expected exactly one needle occurrence, found {matches}"
    )

    needle_pos = matches[0]
    needle_idx = set(
        range(needle_pos, needle_pos + len(needle_ids))
    )

    shuffled_ids = ordered_ids.copy()

    # Shuffle every token except the needle itself.
    movable_idx = [
        i for i in range(len(ordered_ids))
        if i not in needle_idx
    ]

    movable_tokens = [ordered_ids[i] for i in movable_idx]

    rng = np.random.default_rng(ai.SEED + distance)
    rng.shuffle(movable_tokens)

    for i, token_id in zip(movable_idx, movable_tokens):
        shuffled_ids[i] = token_id

    # Critical structural assertions.
    assert len(shuffled_ids) == len(ordered_ids)
    assert shuffled_ids[needle_pos:needle_pos + len(needle_ids)] == needle_ids

    return spec, ordered_ids, shuffled_ids, needle_pos


checks = []

for needle in needles:
    for dist in (1024, 4096):
        spec, ordered_ids, shuffled_ids, needle_pos = make_c3_fixed_shuffle(
            needle, dist
        )

        checks.append({
            "needle": needle,
            "distance": dist,
            "n_tokens": len(ordered_ids),
            "needle_pos": needle_pos,
            "needle_unchanged": (
                ordered_ids[needle_pos] == shuffled_ids[needle_pos]
            ),
            "same_token_multiset": (
                sorted(ordered_ids) == sorted(shuffled_ids)
            ),
        })

pd.DataFrame(checks)

,needle,distance,n_tokens,needle_pos,needle_unchanged,same_token_multiset
0,Paris,1024,9252,148,True,True
1,Paris,4096,12324,148,True,True
2,banana,1024,9252,148,True,True
3,banana,4096,12324,148,True,True
4,Tokyo,1024,9252,148,True,True
5,Tokyo,4096,12324,148,True,True
6,violin,1024,9252,148,True,True
7,violin,4096,12324,148,True,True
8,cinnamon,1024,9252,148,True,True
9,cinnamon,4096,12324,148,True,True


In [38]:
# C3 — matched ordered vs fixed-position shuffled J-Lens run

c3_rows = []

for needle, needle_id in needles.items():
    for dist in (1024, 4096):

        spec, ordered_ids, shuffled_ids, needle_pos = make_c3_fixed_shuffle(
            needle, dist, filler_idx=0
        )

        for condition, input_ids in [
            ("ordered", ordered_ids),
            ("shuffled", shuffled_ids),
        ]:
            ins = {
    "input_ids": torch.tensor(
        [input_ids],
        dtype=torch.long,
        device=bundle.model.device,
    )
}
            out = probe.run(
                ins,
                nowrite=False,
                layers=EXP["layers"],
                capture_residual=False,
            )

            for L in EXP["layers"]:
                if L not in out.ahn_raw:
                    continue

                o_t = out.o_t(L, pos=-1)

                logits = ai.readout_logits(
                    o_t,
                    bundle,
                    lens=lens,
                    layer=L,
                )

                c3_rows.append({
                    "needle": needle,
                    "requested_distance": dist,
                    "filler_idx": 0,
                    "layer": L,
                    "condition": condition,
                    "n_tokens": len(input_ids),
                    "needle_pos": needle_pos,
                    "rank": ai.token_rank(logits, needle_id),
                    "p_mem": ai.token_prob(logits, needle_id),
                    "o_t_norm": float(o_t.float().norm()),
                    "readout": READOUT,
                    "lens_validated": LENS_VALIDATED,
                })

            del out
            ai.free_cuda()

        print(f"done: {needle} @ {dist}")

print("C3 rows:", len(c3_rows))

done: Paris @ 1024
done: Paris @ 4096
done: banana @ 1024
done: banana @ 4096
done: Tokyo @ 1024
done: Tokyo @ 4096
done: violin @ 1024
done: violin @ 4096
done: cinnamon @ 1024
done: cinnamon @ 4096
done: harbour @ 1024
done: harbour @ 4096
done: lantern @ 1024
done: lantern @ 4096
done: trumpet @ 1024
done: trumpet @ 4096
C3 rows: 96


### C3 run completion

The corrected C3 rerun completed successfully for all 8 needles at both tested eviction distances (1024 and 4096).

A total of 96 rows were collected:
- 8 needles
- 2 distances
- 2 conditions: ordered and shuffled
- 3 J-Lens layers: 9, 18, and 27

No runs were dropped during this sweep.

In [39]:
# C3 — verify matched ordered/shuffled pairs before analysis

c3_df = pd.DataFrame(c3_rows)

print("Rows:", len(c3_df))
print("\nRows per condition:")
print(c3_df["condition"].value_counts())

print("\nRows per layer/condition:")
print(
    c3_df.groupby(["layer", "condition"])
    .size()
    .unstack(fill_value=0)
)

pair_counts = (
    c3_df.groupby(
        ["needle", "requested_distance", "filler_idx", "layer"]
    )["condition"]
    .nunique()
)

print("\nMatched pairs:", (pair_counts == 2).sum())
print("Incomplete pairs:", (pair_counts != 2).sum())

assert len(c3_df) == 96
assert set(c3_df["condition"]) == {"ordered", "shuffled"}
assert (pair_counts == 2).all()

print("\nC3 MATCHING CHECK: PASS")

Rows: 96

Rows per condition:
condition
ordered     48
shuffled    48
Name: count, dtype: int64

Rows per layer/condition:
condition  ordered  shuffled
layer                       
9               16        16
18              16        16
27              16        16

Matched pairs: 48
Incomplete pairs: 0

C3 MATCHING CHECK: PASS


### C3 matching check

The corrected C3 dataset is fully balanced and matched.

There are 96 total rows:
- 48 ordered
- 48 shuffled
- 16 ordered and 16 shuffled rows at each layer
- 48 complete ordered–shuffled pairs
- 0 incomplete pairs

Therefore, the C3 statistical comparison can proceed using matched pairs.

In [40]:
# C3 — DIAGNOSTIC pooled t-test across distances
# Superseded for final inference by the distance-stratified exact sign-flip analysis below.

pair_cols = [
    "needle",
    "requested_distance",
    "filler_idx",
    "layer",
]

paired = (
    c3_df.pivot(
        index=pair_cols,
        columns="condition",
        values=["rank", "p_mem"],
    )
    .reset_index()
)

results = []

for layer in EXP["layers"]:
    x = paired[paired["layer"] == layer].copy()

    # Positive rank_delta = shuffling made rank worse.
    rank_delta = (
        x[("rank", "shuffled")].to_numpy(dtype=float)
        - x[("rank", "ordered")].to_numpy(dtype=float)
    )

    # Positive log-prob effect = ordered context gives higher needle probability.
    eps = np.finfo(float).tiny
    log_prob_effect = np.log(
        np.maximum(x[("p_mem", "ordered")].to_numpy(dtype=float), eps)
        / np.maximum(x[("p_mem", "shuffled")].to_numpy(dtype=float), eps)
    )

    rank_t, rank_p = stats.ttest_1samp(rank_delta, 0.0)
    prob_t, prob_p = stats.ttest_1samp(log_prob_effect, 0.0)

    results.append({
        "layer": layer,
        "n_pairs": len(x),

        "mean_rank_ordered": x[("rank", "ordered")].mean(),
        "mean_rank_shuffled": x[("rank", "shuffled")].mean(),
        "mean_rank_delta_shuf_minus_ord": rank_delta.mean(),
        "rank_t": rank_t,
        "rank_p": rank_p,

        "geom_prob_ratio_ord_over_shuf": np.exp(log_prob_effect.mean()),
        "prob_t": prob_t,
        "prob_p": prob_p,
    })

c3_results = pd.DataFrame(results)

print(c3_results.to_string(index=False))

 layer  n_pairs  mean_rank_ordered  mean_rank_shuffled  mean_rank_delta_shuf_minus_ord    rank_t   rank_p  geom_prob_ratio_ord_over_shuf    prob_t   prob_p
     9       16        103256.6875         104283.1875                       1026.5000  0.107367 0.915921                       0.152723 -2.191489 0.044614
    18       16        136610.8750          59775.6250                     -76835.2500 -7.868997 0.000001                     105.459658  2.499371 0.024536
    27       16         75148.3750          51948.8125                     -23199.5625 -3.152357 0.006575                       0.013934 -6.246810 0.000016


### C3 — Interim interpretation

The corrected C3 experiment successfully matched ordered and shuffled conditions while keeping the needle at the same token position.

The initial paired analysis does not yet support a clean conclusion. Layer 9 shows little change in rank, while Layers 18 and 27 show substantially better ranks after shuffling. However, the probability-based results do not fully agree with the rank-based results, especially at Layer 18.

Because rank and probability disagree, these results should not yet be interpreted as evidence that shuffling helps or hurts retention. The matched-pair diagnostic below is used to determine whether the disagreement is caused by very small probabilities, extreme individual cases, or a broader systematic effect.



In [43]:
# C3 — inspect every matched pair before interpretation

diag = paired.copy()

# Flatten the columns created by pivot()
diag.columns = [
    col[0] if col[1] == "" else f"{col[0]}_{col[1]}"
    for col in diag.columns
]

diag["rank_delta"] = (
    diag["rank_shuffled"]
    - diag["rank_ordered"]
)

eps = np.finfo(float).tiny

diag["log_prob_ratio"] = np.log(
    np.maximum(diag["p_mem_ordered"], eps)
    / np.maximum(diag["p_mem_shuffled"], eps)
)

for layer in EXP["layers"]:
    x = diag[diag["layer"] == layer]

    print("\n" + "=" * 70)
    print(f"LAYER {layer}")
    print("=" * 70)

    print(
        x[
            [
                "needle",
                "requested_distance",
                "rank_ordered",
                "rank_shuffled",
                "rank_delta",
                "p_mem_ordered",
                "p_mem_shuffled",
                "log_prob_ratio",
            ]
        ].to_string(index=False)
    )

    print("\nMedian rank delta:", x["rank_delta"].median())
    print(
        "Rank worse after shuffle:",
        int((x["rank_delta"] > 0).sum()),
        "/",
        len(x),
    )
    print(
        "Probability higher ordered:",
        int((x["log_prob_ratio"] > 0).sum()),
        "/",
        len(x),
    )


LAYER 9
  needle  requested_distance  rank_ordered  rank_shuffled  rank_delta  p_mem_ordered  p_mem_shuffled  log_prob_ratio
   Paris                1024       91374.0       125538.0     34164.0   1.267144e-24    8.044332e-25        0.454383
   Paris                4096       77444.0        96101.0     18657.0   2.158396e-23    2.012569e-22       -2.232632
   Tokyo                1024       63372.0        88223.0     24851.0   3.174592e-24    3.757468e-24       -0.168566
   Tokyo                4096       77312.0        34879.0    -42433.0   2.402261e-23    4.033016e-21       -5.123274
  banana                1024      142733.0       148499.0      5766.0   7.702736e-26    2.429595e-26        1.153851
  banana                4096      140229.0       139404.0      -825.0   5.326150e-25    1.439689e-23       -3.296969
cinnamon                1024      137920.0       104538.0    -33382.0   1.306628e-25    2.601519e-24       -2.991231
cinnamon                4096      131710.0       123626

### C3 — Matched-pair diagnostic

The pair-level diagnostic shows that the disagreement between rank and
probability at Layer 18 is systematic rather than being driven by a small
number of extreme observations.

Layer 9 shows no consistent directional effect.

At Layer 18, shuffled context improves needle rank in all 16 matched pairs.
However, the probability effect depends strongly on eviction distance:
shuffling increases needle probability for every 1024-distance pair, whereas
the ordered condition has higher probability for every 4096-distance pair.

At Layer 27, shuffling generally improves the needle readout: shuffled rank is
better in 13 of 16 pairs and shuffled probability is higher in all 16 pairs.

Because Layer 18 shows a clear distance-dependent interaction, the next
analysis evaluates C3 separately at 1024 and 4096 tokens before making a final
control conclusion.

In [45]:
# C3 — stratified matched analysis by layer × eviction distance

c3_stratified = []

for layer in EXP["layers"]:
    for distance in (1024, 4096):

        x = diag[
            (diag["layer"] == layer)
            & (diag["requested_distance"] == distance)
        ].copy()

        assert len(x) == 8, (
            f"Expected 8 matched pairs for layer={layer}, distance={distance}, "
            f"got {len(x)}"
        )

        rank_delta = x["rank_delta"].to_numpy(dtype=float)
        log_prob_ratio = x["log_prob_ratio"].to_numpy(dtype=float)

        c3_stratified.append({
            "layer": layer,
            "distance": distance,
            "n_pairs": len(x),

            # Positive = shuffling made rank worse.
            "mean_rank_delta": rank_delta.mean(),
            "median_rank_delta": np.median(rank_delta),
            "rank_hurt_by_shuffle": int((rank_delta > 0).sum()),

            # >1 = ordered context had higher needle probability.
            "geom_prob_ratio_ord_over_shuf": np.exp(
                log_prob_ratio.mean()
            ),
            "median_log_prob_ratio": np.median(log_prob_ratio),
            "prob_higher_ordered": int(
                (log_prob_ratio > 0).sum()
            ),
        })

c3_stratified = pd.DataFrame(c3_stratified)

print(c3_stratified.to_string(index=False))

 layer  distance  n_pairs  mean_rank_delta  median_rank_delta  rank_hurt_by_shuffle  geom_prob_ratio_ord_over_shuf  median_log_prob_ratio  prob_higher_ordered
     9      1024        8        -5544.375             -226.0                     4                       0.190511              -0.638790                    3
     9      4096        8         7597.375             2287.5                     4                       0.122430              -2.935305                    1
    18      1024        8      -104273.750          -102259.5                     0                       0.105952              -1.664091                    0
    18      4096        8       -49396.750           -48575.5                     0                  104969.397217              12.190050                    8
    27      1024        8       -28309.250           -24912.0                     2                       0.008241              -4.210376                    0
    27      4096        8       -18089.875    

### C3 — Distance-stratified interpretation

The C3 effect depends strongly on both layer and eviction distance.

At Layer 9, there is no consistent rank effect at either distance. Shuffling improves needle probability on average at both 1024 and 4096 tokens, but the rank results are evenly split across matched pairs.

At Layer 18, shuffling improves needle rank in every matched pair at both distances. However, the probability effect reverses with distance. At 1024 tokens, shuffled context produces substantially higher needle probability (ordered/shuffled geometric ratio ≈ 0.106). At 4096 tokens, ordered context produces substantially higher needle probability (ratio ≈ 104,969).

At Layer 27, shuffling improves both rank and probability at both distances. The effect is especially consistent in probability, where shuffled context has higher needle probability in all matched pairs.

Therefore, C3 does not show a simple pattern in which destroying context order uniformly harms the retained needle signal. Layer 18 shows a strong distance-dependent reversal in probability, while Layer 27 generally improves after shuffling.

These results are still exploratory until the distance-stratified effects are tested statistically.

### C3 — Exact-test resolution caveat

Each layer × distance test contains 8 matched pairs.

For a two-sided exact sign-flip test with 8 pairs, there are only \(2^8 = 256\)
possible sign assignments. Therefore, the smallest possible exact p-value is
\(2/256 = 0.0078125\).

After Holm correction across the 6 layer × distance tests, the smallest possible
adjusted p-value is 0.046875.

Therefore, a Holm-adjusted p-value of 0.046875 should be read as
**all 8 matched pairs agreeing in direction**, not as a finely resolved estimate
of statistical significance.

In [46]:
# C3 — exact matched statistical tests by layer × eviction distance

def exact_signflip_pvalue(values):
    values = np.asarray(values, dtype=float)
    n = len(values)

    assert n > 0

    observed = abs(values.mean())

    # Enumerate all 2^n possible sign assignments.
    masks = (
        (np.arange(2 ** n)[:, None] >> np.arange(n))
        & 1
    )
    signs = 2 * masks - 1

    null_stats = np.abs(
        (signs * values[None, :]).mean(axis=1)
    )

    return float(
        np.mean(null_stats >= observed - 1e-12)
    )


def holm_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    m = len(pvalues)

    order = np.argsort(pvalues)
    adjusted = np.empty(m, dtype=float)

    running_max = 0.0

    for i, idx in enumerate(order):
        value = (m - i) * pvalues[idx]
        running_max = max(running_max, value)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted


tests = []

for layer in EXP["layers"]:
    for distance in (1024, 4096):

        x = diag[
            (diag["layer"] == layer)
            & (diag["requested_distance"] == distance)
        ]

        assert len(x) == 8

        rank_delta = x["rank_delta"].to_numpy(dtype=float)
        log_prob_ratio = x["log_prob_ratio"].to_numpy(dtype=float)

        tests.append({
            "layer": layer,
            "distance": distance,
            "n_pairs": len(x),

            # Positive = ordered better / shuffle hurts.
            "mean_rank_delta": rank_delta.mean(),
            "rank_p_exact": exact_signflip_pvalue(rank_delta),

            # Positive = ordered has higher needle probability.
            "mean_log_prob_ratio": log_prob_ratio.mean(),
            "geom_prob_ratio_ord_over_shuf":
                np.exp(log_prob_ratio.mean()),
            "prob_p_exact":
                exact_signflip_pvalue(log_prob_ratio),
        })


c3_tests = pd.DataFrame(tests)

# Correct the six layer × distance tests separately
# for the rank and probability families.
c3_tests["rank_p_holm"] = holm_adjust(
    c3_tests["rank_p_exact"].to_numpy()
)

c3_tests["prob_p_holm"] = holm_adjust(
    c3_tests["prob_p_exact"].to_numpy()
)

print(c3_tests.to_string(index=False))

 layer  distance  n_pairs  mean_rank_delta  rank_p_exact  mean_log_prob_ratio  geom_prob_ratio_ord_over_shuf  prob_p_exact  rank_p_holm  prob_p_holm
     9      1024        8        -5544.375      0.726562            -1.658043                       0.190511      0.203125     1.000000     0.328125
     9      4096        8         7597.375      0.734375            -2.100217                       0.122430      0.164062     1.000000     0.328125
    18      1024        8      -104273.750      0.007812            -2.244767                       0.105952      0.007812     0.046875     0.046875
    18      4096        8       -49396.750      0.007812            11.561424                  104969.397217      0.007812     0.046875     0.046875
    27      1024        8       -28309.250      0.078125            -4.798606                       0.008241      0.007812     0.281250     0.046875
    27      4096        8       -18089.875      0.070312            -3.748186                       0.0235

### C3 — Final conclusion

After correcting the shuffled-context control so that the needle remains at
the same token position, C3 does not support the original expectation that
destroying context order should uniformly weaken the retained needle signal.

Layer 9 shows no statistically reliable rank effect in either direction after
Holm correction.

Layer 18 shows a strong but distance-dependent pattern. At both 1024 and 4096
tokens, shuffling improves needle rank in all 8 matched pairs at each distance.
However, needle probability reverses direction with distance: shuffled context
has higher probability at 1024 tokens, whereas ordered context has higher
probability at 4096 tokens.

Layer 27 shows consistently higher needle probability after shuffling at both
distances, while the corresponding rank effects do not survive Holm correction.

Therefore, the corrected C3 control does **not** show that preserving word order
is generally required for the observed needle readout. The effect of shuffling
depends on layer, eviction distance, and readout metric.

This result should **not** be interpreted as evidence that shuffling improves
AHN memory. Instead, C3 fails to provide the expected order-sensitive control
needed to support a simple content-memory interpretation of the readout.

All C3 conclusions remain conditional on the J-Lens not having passed the full
Table 3 validation battery.